In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 1 — Setup and Project Paths (canonical)
# Purpose: mount Drive, define ALL project paths, import EDA libraries
# ============================================================


import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- The ONE root. Edit only this line if your Drive differs. ---
PROJECT_ROOT = os.path.join(DATA_DIR, "..")

# --- Every project folder, matching your Drive exactly ---
DATA_DIR             = os.path.join(PROJECT_ROOT, 'data')
EXPERIMENTS_DIR      = os.path.join(PROJECT_ROOT, 'experiments')
FIGURES_DIR          = os.path.join(PROJECT_ROOT, 'figures')
NOTEBOOKS_DIR        = os.path.join(PROJECT_ROOT, 'notebooks')
PAPERS_DIR           = os.path.join(PROJECT_ROOT, 'papers')
PROJECT_DECISION_DIR = os.path.join(PROJECT_ROOT, 'project_decision')
RESEARCH_JOURNAL_DIR = os.path.join(PROJECT_ROOT, 'research_journal')
RESULTS_DIR          = os.path.join(PROJECT_ROOT, 'results')
ROADMAP_DIR          = os.path.join(PROJECT_ROOT, 'RoadMap')
SRC_DIR              = os.path.join(PROJECT_ROOT, 'src')
THESIS_NOTES_DIR     = os.path.join(PROJECT_ROOT, 'thesis_notes')

# --- Make src/ importable ---
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# --- Confirm folders exist ---
for name, path in {
    "data": DATA_DIR, "experiments": EXPERIMENTS_DIR, "figures": FIGURES_DIR,
    "notebooks": NOTEBOOKS_DIR, "papers": PAPERS_DIR,
    "project_decision": PROJECT_DECISION_DIR, "research_journal": RESEARCH_JOURNAL_DIR,
    "results": RESULTS_DIR, "RoadMap": ROADMAP_DIR, "src": SRC_DIR,
    "thesis_notes": THESIS_NOTES_DIR,
}.items():
    print(f"{name:18s} {'OK ' if os.path.isdir(path) else 'MISSING'}  {path}")

In [ ]:
import pandas as pd
cols = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
df = pd.read_csv('{DATA_DIR}/../data/train_FD001.txt', sep=r'\s+', header=None, names=cols)
print(df[df.unit.isin([1,2])].groupby('unit').head(3)[['unit','cycle','s2','s3','s4','s11']])

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 1 — Setup and Project Paths (canonical)
# Purpose: mount Drive, define ALL project paths, import EDA libraries
# ============================================================


import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- The ONE root. Edit only this line if your Drive differs. ---
PROJECT_ROOT = os.path.join(DATA_DIR, "..")

# --- Every project folder, matching your Drive exactly ---
DATA_DIR             = os.path.join(PROJECT_ROOT, 'data')
EXPERIMENTS_DIR      = os.path.join(PROJECT_ROOT, 'experiments')
FIGURES_DIR          = os.path.join(PROJECT_ROOT, 'figures')
NOTEBOOKS_DIR        = os.path.join(PROJECT_ROOT, 'notebooks')
PAPERS_DIR           = os.path.join(PROJECT_ROOT, 'papers')
PROJECT_DECISION_DIR = os.path.join(PROJECT_ROOT, 'project_decision')
RESEARCH_JOURNAL_DIR = os.path.join(PROJECT_ROOT, 'research_journal')
RESULTS_DIR          = os.path.join(PROJECT_ROOT, 'results')
ROADMAP_DIR          = os.path.join(PROJECT_ROOT, 'RoadMap')
SRC_DIR              = os.path.join(PROJECT_ROOT, 'src')
THESIS_NOTES_DIR     = os.path.join(PROJECT_ROOT, 'thesis_notes')

# --- Make src/ importable ---
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

# --- Confirm folders exist ---
for name, path in {
    "data": DATA_DIR, "experiments": EXPERIMENTS_DIR, "figures": FIGURES_DIR,
    "notebooks": NOTEBOOKS_DIR, "papers": PAPERS_DIR,
    "project_decision": PROJECT_DECISION_DIR, "research_journal": RESEARCH_JOURNAL_DIR,
    "results": RESULTS_DIR, "RoadMap": ROADMAP_DIR, "src": SRC_DIR,
    "thesis_notes": THESIS_NOTES_DIR,
}.items():
    print(f"{name:18s} {'OK ' if os.path.isdir(path) else 'MISSING'}  {path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cols = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
df = pd.read_csv('{DATA_DIR}/../data/train_FD001.txt', sep=r'\s+', header=None, names=cols)

life = df.groupby('unit')['cycle'].max().sort_values()
engines = [life.index[0], life.index[len(life)//2], life.index[-1]]

sensors = ['s11','s12','s4','s9']
titles  = ['s11 — static pressure at HPC outlet',
           's12 — ratio of fuel flow to Ps30',
           's4  — LPT outlet temperature',
           's9  — physical core speed']

fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
for ax, s, t in zip(axes.ravel(), sensors, titles):
    for e in engines:
        d = df[df.unit == e]
        ax.plot(d.cycle, d[s], lw=0.9, label=f'Engine {e} ({life[e]} cycles)')
    ax.set_title(t, fontsize=10)
    ax.set_xlabel('Operating cycle')
    ax.set_ylabel(s)
axes[0, 0].legend(fontsize=8, frameon=False)
fig.savefig('fd001_raw_trajectories.png', dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 11, 'axes.labelsize': 10})

cols = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
df = pd.read_csv('{DATA_DIR}/../data/train_FD001.txt',
                 sep=r'\s+', header=None, names=cols)

life = df.groupby('unit')['cycle'].max().sort_values()
engines = [life.index[0], life.index[len(life)//2], life.index[-1]]
colours = ['#08306b', '#d94801', '#41ab5d']     # dark / mid / light

panels = [('s11', 'Static pressure at HPC outlet (s11)', 'psia'),
          ('s12', 'Ratio of fuel flow to Ps30 (s12)',    'pps/psi'),
          ('s4',  'LPT outlet temperature (s4)',         '°R'),
          ('s9',  'Physical core speed (s9)',            'rpm')]

fig, axes = plt.subplots(2, 2, figsize=(9, 6.2), constrained_layout=True)
for ax, (s, title, unit) in zip(axes.ravel(), panels):
    for e, c in zip(engines, colours):
        d = df[df.unit == e]
        ax.plot(d.cycle, d[s], lw=0.8, color=c,
                label=f'Engine {e} — {life[e]} cycles')
    ax.set_title(title)
    ax.set_xlabel('Operating cycle')
    ax.set_ylabel(unit)
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=3, frameon=False)

fig.savefig('{DATA_DIR}/../fd001_raw_trajectories.png',
            dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 11, 'axes.labelsize': 10})

cols = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
df = pd.read_csv('{DATA_DIR}/../data/train_FD001.txt',
                 sep=r'\s+', header=None, names=cols)

life = df.groupby('unit')['cycle'].max().sort_values()
engines = [life.index[0], life.index[len(life)//2], life.index[-1]]
colours = ['#08306b', '#d94801', '#41ab5d']     # dark / mid / light

panels = [('s11', 'Static pressure at HPC outlet (s11)', 'psia'),
          ('s12', 'Ratio of fuel flow to Ps30 (s12)',    'pps/psi'),
          ('s4',  'LPT outlet temperature (s4)',         '°R'),
          ('s9',  'Physical core speed (s9)',            'rpm')]

fig, axes = plt.subplots(2, 2, figsize=(9, 6.2), constrained_layout=True)
for ax, (s, title, unit) in zip(axes.ravel(), panels):
    for e, c in zip(engines, colours):
        d = df[df.unit == e]
        ax.plot(d.cycle, d[s], lw=0.8, color=c,
                label=f'Engine {e} — {life[e]} cycles')
    ax.set_title(title)
    ax.set_xlabel('Operating cycle')
    ax.set_ylabel(unit)
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=3, frameon=False)

fig.savefig('{DATA_DIR}/../fd001_raw_trajectories.png',
            dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8,
                     'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5,
                     'legend.fontsize': 8})

cols = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
df = pd.read_csv('{DATA_DIR}/../data/train_FD001.txt',
                 sep=r'\s+', header=None, names=cols)

life = df.groupby('unit')['cycle'].max().sort_values()
engines = [life.index[0], life.index[len(life)//2], life.index[-1]]

# one hue, light to dark, ordered by lifetime: the shade itself carries the order
colours = ['#6BAED6', '#2171B5', '#08306B']

panels = [('s11', 'Static pressure at HPC outlet (s11)', 'psia'),
          ('s12', 'Ratio of fuel flow to Ps30 (s12)',    'pps/psi'),
          ('s4',  'LPT outlet temperature (s4)',         '\u00b0R'),
          ('s9',  'Physical core speed (s9)',            'rpm')]

fig, axes = plt.subplots(2, 2, figsize=(6.5, 4.4), dpi=300,
                         constrained_layout=True)

for ax, (s, title, unit) in zip(axes.ravel(), panels):
    for e, c in zip(engines, colours):
        d = df[df.unit == e]
        ax.plot(d.cycle, d[s], lw=0.7, color=c)
    ax.set_title(title, pad=4)
    ax.set_xlabel('Operating cycle')
    ax.set_ylabel(unit)
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')
    ax.tick_params(length=2.5, width=0.6)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.spines['left'].set_linewidth(0.6)
    ax.spines['bottom'].set_linewidth(0.6)

tags = ['shortest', 'median', 'longest']
handles = [Line2D([0], [0], color=c, lw=3.2, solid_capstyle='butt')
           for c in colours]
labels  = [f'Engine {e} \u2014 {life[e]} cycles ({t})'
           for e, t in zip(engines, tags)]

fig.legend(handles, labels, loc='outside lower center', ncol=3,
           frameon=False, handlelength=2.2, handletextpad=0.6,
           columnspacing=2.0)

fig.savefig('{DATA_DIR}/../fd001_raw_trajectories.png',
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd, os
p = "{DATA_DIR}/../results/fd001_importance_sensor.csv"
d = pd.read_csv(p).sort_values("importance", ascending=False)
for _, r in d.iterrows():
    print(f"{r['group']:<6} {r['importance']:6.2f} {r['std']:6.2f}")

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 2 — Load FD001 and Verify
# Purpose: load data via src/load_data.py and confirm it is correct
# ============================================================
import sys

sys.path.append(SRC_DIR)
from load_data import load_cmapss_subset   # our reusable loader from src/

# --- Load the FD001 subset from the data/ folder ---
train, test, y_test = load_cmapss_subset(DATA_DIR, subset="FD001")

# --- Verification checks (Phase 1 "done-when" criteria) ---
print("train shape:", train.shape)               # expect (20631, 26)
print("test  shape:", test.shape)                # expect (13096, 26)
print("RUL   shape:", y_test.shape)              # expect (100, 1)
print("train engines:", train["unit"].nunique()) # expect 100
print("test  engines:", test["unit"].nunique())  # expect 100
print("missing values in train:", train.isna().sum().sum())  # expect 0
print()

train.head()

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 3 — Column Inspection and Constant-Sensor Identification
# Purpose: find sensors/settings with zero (or near-zero) variability.
#          IDENTIFY here; removal happens in 02_Preprocessing.
# Paths: uses RESULTS_DIR from the Section 1 path block (already defined).
# ============================================================

# --- Make sure results/ exists (RESULTS_DIR comes from Section 1) ---
os.makedirs(RESULTS_DIR, exist_ok=True)

# --- Overall descriptive statistics for all numeric columns ---
desc = train.describe().T   # transpose -> one row per column, easier to read
print("Descriptive statistics (mean / std / min / max):")
print(desc[["mean", "std", "min", "max"]].round(3).head(30))
print()

# --- Identify constant columns via standard deviation ---
# A truly constant column has std = 0. The tiny threshold also catches
# sensors that are flat apart from numerical noise.
feature_cols = ["op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
std_values = train[feature_cols].std()

CONST_THRESHOLD = 1e-6
constant_cols = std_values[std_values < CONST_THRESHOLD].index.tolist()

print("Standard deviation per feature:")
print(std_values.round(6))
print()
print(f"Constant / non-informative columns (std < {CONST_THRESHOLD}):")
print(constant_cols)
print(f"Count: {len(constant_cols)}")

# --- Save the finding to results/ ---
std_table = std_values.round(6).reset_index()
std_table.columns = ["feature", "std"]
std_table["is_constant"] = std_table["std"] < CONST_THRESHOLD

save_path = os.path.join(RESULTS_DIR, "fd001_feature_std.csv")
std_table.to_csv(save_path, index=False)
print(f"\nSaved standard-deviation table to:\n{save_path}")

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 3 — Data Quality Checks
# Purpose: explicitly audit FD001 for column dtypes, missing values,
#          duplicates, and outliers. Document the decision NOT to remove outliers.
# Paths: uses FIGURES_DIR and RESULTS_DIR from Section 1.
# ============================================================

# ----- 0. Column dtypes (must all be numeric) -----
# Why: if any column accidentally loaded as text (string), every later
# statistical step (std, corr, IQR) would silently fail or produce nonsense.
# Categorical / object dtypes are not expected anywhere in FD001 — every
# column is either an integer (unit, cycle) or a float (settings, sensors).
print("=== Column dtypes ===")
print(train.dtypes.to_string())
print()

# Explicit pass/fail check
non_numeric = [c for c in train.columns
               if not pd.api.types.is_numeric_dtype(train[c])]
if non_numeric:
    raise TypeError(
        f"Non-numeric columns detected in train: {non_numeric}. "
        "Investigate the source files — possible parsing problem."
    )

# Also confirm test has identical dtypes (defensive — train and test must match)
mismatched = [c for c in train.columns
              if train[c].dtype != test[c].dtype]
if mismatched:
    raise TypeError(
        f"Train/test dtype mismatch on columns: {mismatched}. "
        "Both files must parse to identical dtypes for fair preprocessing."
    )

print(f"All {len(train.columns)} train columns are numeric.")
print(f"Train and test dtypes match on every column.")
print()

# ----- 1. Missing values (re-confirmation across all three files) -----
print("=== Missing values ===")
print(f"train: {train.isna().sum().sum()}")
print(f"test:  {test.isna().sum().sum()}")
print(f"y_test:{y_test.isna().sum().sum()}")
print()

# ----- 2. Duplicate rows -----
# A "duplicate" in this dataset would mean two identical sensor readings
# for the same engine at the same cycle. Even partial duplicates
# (same unit + cycle) would indicate a data issue.
print("=== Duplicate rows ===")
print(f"train: full-row duplicates    = {train.duplicated().sum()}")
print(f"train: (unit, cycle) duplicates = "
      f"{train.duplicated(subset=['unit','cycle']).sum()}")
print(f"test:  full-row duplicates    = {test.duplicated().sum()}")
print(f"test:  (unit, cycle) duplicates = "
      f"{test.duplicated(subset=['unit','cycle']).sum()}")
print()

# ----- 3. Outlier inspection per sensor (IQR rule, INFORMATIONAL ONLY) -----
# We compute how many points would be flagged as "outliers" by the
# standard 1.5*IQR rule, per sensor. We do NOT remove them.
# Reason: in run-to-failure data the late-life extremes are the
# degradation signal itself.
print("=== Outlier counts per sensor (1.5*IQR rule, informational) ===")
sensor_cols = [f"s{i}" for i in range(1, 22)]

outlier_summary = []
for col in sensor_cols:
    q1 = train[col].quantile(0.25)
    q3 = train[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((train[col] < lower) | (train[col] > upper)).sum()
    pct = 100 * n_out / len(train)
    outlier_summary.append({
        "sensor": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_fence": lower, "upper_fence": upper,
        "n_outliers": n_out, "pct_outliers": round(pct, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df[["sensor", "n_outliers", "pct_outliers"]].to_string(index=False))

# --- Save the outlier report to results/ ---
outlier_path = os.path.join(RESULTS_DIR, "fd001_outlier_report.csv")
outlier_df.to_csv(outlier_path, index=False)
print(f"\nSaved outlier report to:\n{outlier_path}")

# ----- 4. Visual check: where do the "outliers" sit in the engine life? -----
# Pick the trending sensor with the most outliers and plot its values
# colored by RUL. If the "outliers" cluster at low RUL (near failure),
# they ARE the degradation signal and must NOT be removed.
trending_sensors = ["s2","s3","s4","s7","s8","s11","s12","s13","s15","s17","s20","s21"]
candidates = outlier_df[outlier_df["sensor"].isin(trending_sensors)]
top_sensor = candidates.sort_values("n_outliers", ascending=False).iloc[0]["sensor"]

# temporary RUL for color-coding only (not saved as a label)
max_cycle = train.groupby("unit")["cycle"].transform("max")
rul_tmp = max_cycle - train["cycle"]

fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(train["cycle"], train[top_sensor],
                c=rul_tmp, cmap="viridis_r", s=4, alpha=0.5)
ax.set_xlabel("cycle")
ax.set_ylabel(top_sensor)
plt.colorbar(sc, label="RUL (cycles to failure)")
fig.tight_layout()

fig_path = os.path.join(FIGURES_DIR, f"fd001_outliers_are_signal_{top_sensor}.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved figure to:\n{fig_path}")

Four integrity checks were applied to both files, each addressing a condition that would change how the later analysis must be carried out. Missing values would call for imputation, and an imputed sensor reading inside a degradation trajectory is a value the engine never produced. Fully duplicated rows would give some cycles more weight than others in the row-level statistics of Section 5.7. Duplicated (unit, cycle) pairs would mean that a cycle index does not identify a single position in a trajectory, which the history-based features of Section 6.4.2 assume it does. Non-numeric entries would force a type conversion whose rule would have to be stated and defended.

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 4 — Sensor Trajectories for One Engine
# Purpose: visualize flat vs trending sensors over one engine's life.
# Paths: uses FIGURES_DIR from Section 1.
# ============================================================

os.makedirs(FIGURES_DIR, exist_ok=True)

# --- Pick one engine to inspect (unit 1 is the conventional choice) ---

ENGINE_ID = 1
engine = train[train["unit"] == ENGINE_ID]

print(
    f"Engine {ENGINE_ID} runs for "
    f"{engine['cycle'].max()} cycles."
)

# --- The 21 sensor columns ---

sensor_cols = [f"s{i}" for i in range(1, 22)]

# --- Plot each sensor against cycle, in a 7x3 grid ---

fig, axes = plt.subplots(7, 3, figsize=(15, 18))
axes = axes.flatten()

for i, col in enumerate(sensor_cols):
    axes[i].plot(
        engine["cycle"],
        engine[col],
        linewidth=1
    )
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("cycle", fontsize=8)
    axes[i].tick_params(labelsize=7)

fig.tight_layout()

# --- Save the figure to figures/ ---

fig_path = os.path.join(
    FIGURES_DIR,
    f"fd001_sensor_trajectories_engine{ENGINE_ID}.png"
)

fig.savefig(
    fig_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(f"\nSaved figure to:\n{fig_path}")

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 5 — Correlation Heatmap of Sensors
# Purpose: identify highly correlated (redundant) sensors in FD001.
# IDENTIFY here; removal happens in 02_Preprocessing.
# Paths: uses FIGURES_DIR and RESULTS_DIR from Section 1.
# ============================================================

# --- The feature columns (3 op settings + 21 sensors) ---

feature_cols = [
    "op1",
    "op2",
    "op3"
] + [f"s{i}" for i in range(1, 22)]

# --- Compute the Pearson correlation matrix ---

corr = train[feature_cols].corr()

# --- Plot the heatmap ---

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"shrink": 0.8},
    annot_kws={"size": 7},
    ax=ax
)

fig.tight_layout()

# --- Save the figure to figures/ ---

fig_path = os.path.join(
    FIGURES_DIR,
    "fd001_correlation_heatmap.png"
)

fig.savefig(
    fig_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(f"Saved figure to:\n{fig_path}\n")

# --- Identify highly correlated pairs (|r| > 0.95) ---
# Excluding the diagonal

HIGH_CORR_THRESHOLD = 0.95

# Upper triangle only
# Avoid (A, B) and (B, A) duplicates and the diagonal

import numpy as np

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper
    .stack()
    .reset_index()
    .rename(
        columns={
            "level_0": "feature_A",
            "level_1": "feature_B",
            0: "corr"
        }
    )
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["corr"].abs() > HIGH_CORR_THRESHOLD
].sort_values(
    "corr",
    key=abs,
    ascending=False
)

print(
    f"Highly correlated feature pairs "
    f"(|r| > {HIGH_CORR_THRESHOLD}):"
)

print(
    high_corr_pairs.to_string(index=False)
)

print(
    f"\nNumber of redundant pairs: "
    f"{len(high_corr_pairs)}"
)

# --- Save the list of redundant pairs to results/ ---

pairs_path = os.path.join(
    RESULTS_DIR,
    "fd001_high_correlation_pairs.csv"
)

high_corr_pairs.to_csv(
    pairs_path,
    index=False
)

print(
    f"\nSaved correlation-pair list to:\n"
    f"{pairs_path}"
)

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 6 — Engine Lifetime Distribution
# Purpose: show how long engines run before failure -> justify RUL cap = 125.
# Paths: uses FIGURES_DIR and RESULTS_DIR from Section 1.
# ============================================================

# --- One lifetime per engine = the maximum cycle in its trajectory ---
lifetimes = train.groupby("unit")["cycle"].max()
print(f"Lifetime statistics across {len(lifetimes)} training engines:\n")
print(lifetimes.describe().round(1))
print(f"\n25th / 50th / 75th percentile: "
      f"{int(lifetimes.quantile(0.25))} / {int(lifetimes.quantile(0.50))} "
      f"/ {int(lifetimes.quantile(0.75))} cycles")

# --- Plot the histogram ---
RUL_CAP = 125

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(lifetimes, bins=20, color="#4C9BD6", edgecolor="white")
ax.axvline(lifetimes.median(), color="#1E8449", linewidth=1.5,
           linestyle=":", label=f"median = {int(lifetimes.median())}")
ax.set_xlabel("Engine lifetime (cycles to failure)")
ax.set_ylabel("Number of engines")
ax.legend()
fig.tight_layout()

# --- Save the figure ---
fig_path = os.path.join(FIGURES_DIR, "fd001_engine_lifetime_distribution.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved figure to:\n{fig_path}")

# --- Save the lifetimes table too (useful in the thesis appendix) ---
lifetimes_path = os.path.join(RESULTS_DIR, "fd001_engine_lifetimes.csv")
lifetimes.reset_index().rename(columns={"cycle": "lifetime_cycles"}).to_csv(
    lifetimes_path, index=False
)
print(f"Saved lifetimes table to:\n{lifetimes_path}")

# --- How many engines actually live LONGER than the cap? ---
above_cap = (lifetimes > RUL_CAP).sum()
print(f"\nEngines that run longer than the cap ({RUL_CAP}): "
      f"{above_cap} / {len(lifetimes)} "
      f"({above_cap / len(lifetimes):.0%})")

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 7 — Average Sensor Trends Across Multiple Engines
# Purpose: confirm trending sensors trend CONSISTENTLY across the fleet
# by aligning all engines by remaining-cycles-to-failure (RUL).
# Note: the rul_linear column built here is TEMPORARY (EDA-only).
# The real RUL labels are constructed in 02_Preprocessing.
# Paths: uses FIGURES_DIR from Section 1.
# ============================================================

# --- Build a temporary linear RUL column for ALIGNMENT only ---

# For each engine:
# RUL = max_cycle - current_cycle

max_cycle_per_engine = (
    train
    .groupby("unit")["cycle"]
    .transform("max")
)

train_eda = train.copy()

train_eda["rul_linear"] = (
    max_cycle_per_engine
    - train_eda["cycle"]
)

# --- Pick the sensors to inspect ---

# All 21 sensors
# Flat and trending sensors appear in the same view

sensor_cols = [f"s{i}" for i in range(1, 22)]

# --- Average each sensor at each RUL value across all engines ---

# Limit the analysis to the final 200 cycles before failure

LOOKBACK = 200

mean_by_rul = (
    train_eda[
        train_eda["rul_linear"] <= LOOKBACK
    ]
    .groupby("rul_linear")[sensor_cols]
    .mean()
    .sort_index(ascending=False)
)

# --- Plot a 7 × 3 grid ---

fig, axes = plt.subplots(
    7,
    3,
    figsize=(15, 18)
)

axes = axes.flatten()

for i, col in enumerate(sensor_cols):
    axes[i].plot(
        mean_by_rul.index,
        mean_by_rul[col],
        linewidth=1.2,
        color="#2E75B6"
    )

    axes[i].set_title(
        col,
        fontsize=10
    )

    axes[i].set_xlabel(
        "RUL (cycles to failure)",
        fontsize=8
    )

    # Failure on the right
    # Healthy period on the left

    axes[i].invert_xaxis()

    axes[i].tick_params(
        labelsize=7
    )

fig.tight_layout()

# --- Save the figure ---

fig_path = os.path.join(
    FIGURES_DIR,
    "fd001_avg_sensor_trends_by_rul.png"
)

fig.savefig(
    fig_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(
    f"\nSaved figure to:\n"
    f"{fig_path}"
)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH   = '{DATA_DIR}/../data/train_FD001.txt'
FIGURES_DIR = '{DATA_DIR}/..'
LOOKBACK    = 200

cols = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1, 22)]
train = pd.read_csv(DATA_PATH, sep=r'\s+', header=None, names=cols)

train['rul'] = train.groupby('unit')['cycle'].transform('max') - train['cycle']
sub = train[train['rul'] <= LOOKBACK]

sensor_cols = [f's{i}' for i in range(1, 22)]
mean_by_rul = sub.groupby('rul')[sensor_cols].mean()
std_by_rul  = sub.groupby('rul')[sensor_cols].std()
x = mean_by_rul.index

fig, axes = plt.subplots(7, 3, figsize=(12, 19), sharex=True,
                         constrained_layout=True)
axes = axes.flatten()

for ax, col in zip(axes, sensor_cols):
    m, s = mean_by_rul[col], std_by_rul[col]
    ax.fill_between(x, m - s, m + s, color='#9ecae1', alpha=0.5, lw=0,
                    label='±1 standard deviation across engines')
    ax.plot(x, m, lw=1.0, color='#08519c', label='Fleet mean')
    ax.axvline(127, color='0.45', lw=0.7, ls='--',
               label='All 100 engines present to the right of this line')
    ax.set_title(col, fontsize=10, pad=4)
    ax.tick_params(labelsize=7)
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')

axes[0].invert_xaxis()          # shared axis — invert once only

for ax in axes[-3:]:
    ax.set_xlabel('RUL (cycles to failure)', fontsize=9)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=3,
           frameon=False, fontsize=10)

os.makedirs(FIGURES_DIR, exist_ok=True)
out_path = os.path.join(FIGURES_DIR, 'fd001_fleet_trends_with_bands.png')
fig.savefig(out_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved to:', out_path)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH   = '{DATA_DIR}/../data/train_FD001.txt'
FIGURES_DIR = '{DATA_DIR}/..'

cols = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1, 22)]
train = pd.read_csv(DATA_PATH, sep=r'\s+', header=None, names=cols)

life    = train.groupby('unit')['cycle'].max().sort_values()
engines = [life.index[0], life.index[len(life) // 2], life.index[-1]]
colours = ['#08306b', '#d94801', '#41ab5d']

sensor_cols = [f's{i}' for i in range(1, 22)]

fig, axes = plt.subplots(7, 3, figsize=(12, 19), sharex=True,
                         constrained_layout=True)
axes = axes.flatten()

for ax, col in zip(axes, sensor_cols):
    for e, c in zip(engines, colours):
        d = train[train['unit'] == e]
        ax.plot(d['cycle'], d[col], lw=0.6, color=c,
                label=f'Engine {e} — {life[e]} cycles')
    ax.set_title(col, fontsize=10, pad=4)
    ax.tick_params(labelsize=7)
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')
    ax.margins(x=0.02)

# x-label on the bottom row only
for ax in axes[-3:]:
    ax.set_xlabel('Operating cycle', fontsize=9)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=3,
           frameon=False, fontsize=11)

os.makedirs(FIGURES_DIR, exist_ok=True)
out_path = os.path.join(FIGURES_DIR, 'fd001_all_sensors_three_engines.png')
fig.savefig(out_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved to:', out_path)

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 10 — Degradation Pattern Check
# Monotonicity / No Maintenance Events
#
# Purpose: verify FD001 shows monotonic degradation,
# not maintenance-event resets.
#
# Method:
# (a) overlay 15 engines' s11 trajectories for visual inspection
# (b) compute net rise, max drawdown, and fraction-increasing
#     for all engines using a 20-cycle smoothing window
#
# Paths: uses FIGURES_DIR and RESULTS_DIR from Section 1.
# ============================================================

import numpy as np

# --- The trending sensor used for the monotonicity test ---

SENSOR = "s11"
WINDOW = 20

# ============================================================
# Part 1 — Visual overlay of 15 random engines
# ============================================================

np.random.seed(42)

sample_engines = np.random.choice(
    train["unit"].unique(),
    size=15,
    replace=False
)

fig, ax = plt.subplots(figsize=(12, 6))

for engine_id in sample_engines:
    engine = train[
        train["unit"] == engine_id
    ]

    ax.plot(
        engine["cycle"],
        engine[SENSOR],
        linewidth=0.8,
        alpha=0.55,
        label=f"Engine {engine_id}"
    )

ax.set_xlabel("Cycle")

ax.set_ylabel(
    f"{SENSOR} "
    "(Ps30 — static pressure at HPC outlet, psia)"
)

ax.legend(
    fontsize=7,
    ncol=3,
    loc="best"
)

fig.tight_layout()

fig_path = os.path.join(
    FIGURES_DIR,
    f"fd001_monotonicity_check_{SENSOR}.png"
)

fig.savefig(
    fig_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(f"Saved figure to:\n{fig_path}\n")

# ============================================================
# Part 2 — Quantitative monotonicity test
# Three metrics calculated using the smoothed signal
# ============================================================

print(
    f"=== Monotonicity check for {SENSOR} "
    f"(smoothing window = {WINDOW} cycles) ===\n"
)

print("Three metrics per engine on the smoothed signal:")

print(
    "  (A) net rise: end value − start value. "
    "Should be POSITIVE for rising sensors."
)

print(
    "  (B) max drawdown: largest drop from running peak."
)

print(
    "      A maintenance event would cause a drop "
    "comparable to the full trend."
)

print(
    "  (C) fraction of consecutive cycles where "
    "the smoothed value increased.\n"
)

results = []

for engine_id in train["unit"].unique():
    engine = (
        train[
            train["unit"] == engine_id
        ]
        .sort_values("cycle")
    )

    smoothed = (
        engine[SENSOR]
        .rolling(
            window=WINDOW,
            min_periods=WINDOW
        )
        .mean()
        .dropna()
    )

    if len(smoothed) < 2:
        continue

    # (A) Net rise
    # Start-to-end change in the smoothed signal

    net_rise = (
        smoothed.iloc[-1]
        - smoothed.iloc[0]
    )

    # (B) Maximum drawdown
    # Largest drop from the running peak

    drawdown = (
        smoothed.cummax()
        - smoothed
    ).max()

    # (C) Fraction of smoothed cycle-pairs that increased

    diffs = smoothed.diff().dropna()

    frac_up = (
        (diffs > 0).sum()
        / len(diffs)
    )

    results.append(
        {
            "unit": engine_id,
            "lifetime": engine["cycle"].max(),
            "net_rise_psia": round(net_rise, 3),
            "max_drawdown_psia": round(drawdown, 3),
            "frac_cycles_increasing_smoothed": round(
                frac_up,
                3
            )
        }
    )

mono_df = pd.DataFrame(results)

# --- Summary statistics ---

print(f"Across {len(mono_df)} engines:\n")

print("(A) Net rise (end − start, psia):")

print(
    f"    mean   "
    f"{mono_df['net_rise_psia'].mean():.3f}"
)

print(
    f"    median "
    f"{mono_df['net_rise_psia'].median():.3f}"
)

print(
    f"    min    "
    f"{mono_df['net_rise_psia'].min():.3f}"
)

print(
    f"    max    "
    f"{mono_df['net_rise_psia'].max():.3f}"
)

print(
    f"    Engines with NEGATIVE net rise: "
    f"{(mono_df['net_rise_psia'] < 0).sum()} "
    "(expect 0 — sensor should rise overall)\n"
)

print(
    "(B) Maximum drawdown from running peak (psia):"
)

print(
    f"    mean   "
    f"{mono_df['max_drawdown_psia'].mean():.3f}"
)

print(
    f"    median "
    f"{mono_df['max_drawdown_psia'].median():.3f}"
)

print(
    f"    max    "
    f"{mono_df['max_drawdown_psia'].max():.3f}"
)

print(
    "    Interpretation: if drawdown ≈ or > net rise, "
    "a maintenance-event"
)

print(
    "    recovery is likely. If drawdown << net rise, "
    "degradation is monotonic.\n"
)

print(
    "(C) Fraction of smoothed cycle-pairs increasing:"
)

print(
    f"    mean   "
    f"{mono_df['frac_cycles_increasing_smoothed'].mean():.3f}"
)

print(
    f"    median "
    f"{mono_df['frac_cycles_increasing_smoothed'].median():.3f}"
)

print(
    f"    min    "
    f"{mono_df['frac_cycles_increasing_smoothed'].min():.3f}"
)

print(
    f"    max    "
    f"{mono_df['frac_cycles_increasing_smoothed'].max():.3f}"
)

print(
    "    (0.5 = random; 0.65–0.85 = trending; "
    "0.85+ = strongly monotonic)\n"
)

# --- Engines flagged for further inspection ---

flagged = mono_df[
    (mono_df["net_rise_psia"] < 0)
    |
    (
        mono_df["max_drawdown_psia"]
        > mono_df["net_rise_psia"]
    )
]

print(
    "Engines flagged "
    "(negative net rise OR drawdown > net rise): "
    f"{len(flagged)}"
)

if len(flagged) > 0:
    print(
        flagged.to_string(index=False)
    )
else:
    print(
        "  None. All engines show monotonic rising "
        "behavior consistent with"
    )

    print(
        "  pure degradation — no evidence of "
        "maintenance events."
    )

# --- Save the table ---

mono_path = os.path.join(
    RESULTS_DIR,
    f"fd001_monotonicity_{SENSOR}_v2.csv"
)

mono_df.to_csv(
    mono_path,
    index=False
)

print(
    f"\nSaved monotonicity table to:\n"
    f"{mono_path}"
)

# ============================================================
# Part 3 — Companion diagram
# Three smoothed metrics across the fleet
# ============================================================

fig2, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4)
)

# A) Net-rise distribution

axes[0].hist(
    mono_df["net_rise_psia"],
    bins=20,
    color="#2E75B6",
    edgecolor="white"
)

axes[0].axvline(
    0,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="zero (no rise)"
)

axes[0].set_xlabel(
    "Net rise (psia)"
)

axes[0].set_ylabel(
    "Number of engines"
)

axes[0].legend(
    fontsize=8
)

# B) Maximum-drawdown distribution

axes[1].hist(
    mono_df["max_drawdown_psia"],
    bins=20,
    color="#D9822B",
    edgecolor="white"
)

axes[1].set_xlabel(
    "Max drawdown (psia)"
)

axes[1].set_ylabel(
    "Number of engines"
)

# C) Fraction-increasing distribution

axes[2].hist(
    mono_df["frac_cycles_increasing_smoothed"],
    bins=20,
    color="#1E8449",
    edgecolor="white"
)

axes[2].axvline(
    0.5,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="0.5 (random)"
)

axes[2].set_xlabel(
    "Fraction of cycles increasing"
)

axes[2].set_ylabel(
    "Number of engines"
)

axes[2].legend(
    fontsize=8
)

fig2.tight_layout()

fig2_path = os.path.join(
    FIGURES_DIR,
    f"fd001_monotonicity_metrics_{SENSOR}.png"
)

fig2.savefig(
    fig2_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(
    f"\nSaved metrics diagram to:\n"
    f"{fig2_path}"
)

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 11 — Inter-Engine Heterogeneity
# Purpose: quantify how similar/different the 100 engines are in
# starting condition, degradation rate, and lifetime.
# Paths: uses FIGURES_DIR and RESULTS_DIR from Section 1.
# ============================================================

import numpy as np

# --- The trending sensor we use as a degradation proxy ---

SENSOR = "s11"

# ============================================================
# Part 1 — Compute three heterogeneity metrics per engine
# ============================================================

# (1) Starting sensor value
#     Average of first 5 cycles per engine

# (2) Ending sensor value
#     Average of last 5 cycles per engine

# (3) Degradation rate per engine
#     Slope of s11 vs cycle in psia/cycle

heterogeneity = []

for engine_id in train["unit"].unique():
    engine = (
        train[
            train["unit"] == engine_id
        ]
        .sort_values("cycle")
    )

    n = len(engine)

    start_val = (
        engine[SENSOR]
        .iloc[:5]
        .mean()
    )

    end_val = (
        engine[SENSOR]
        .iloc[-5:]
        .mean()
    )

    # Slope via simple linear fit over the entire trajectory

    slope, _ = np.polyfit(
        engine["cycle"].values,
        engine[SENSOR].values,
        1
    )

    heterogeneity.append(
        {
            "unit": engine_id,
            "lifetime": n,
            "start_s11_psia": round(start_val, 3),
            "end_s11_psia": round(end_val, 3),
            "slope_psia_per_cycle": round(slope, 5)
        }
    )

het_df = pd.DataFrame(heterogeneity)

# --- Summary statistics ---

print(
    "=== Inter-engine heterogeneity "
    "(sensor s11 across 100 engines) ===\n"
)

print(
    "Starting condition "
    "(avg of first 5 cycles):"
)

print(
    f"  mean   "
    f"{het_df['start_s11_psia'].mean():.3f} psia"
)

print(
    f"  std    "
    f"{het_df['start_s11_psia'].std():.3f} psia"
)

print(
    f"  min    "
    f"{het_df['start_s11_psia'].min():.3f}"
)

print(
    f"  max    "
    f"{het_df['start_s11_psia'].max():.3f}"
)

print(
    f"  range  "
    f"{het_df['start_s11_psia'].max() - het_df['start_s11_psia'].min():.3f} "
    "psia\n"
)

print(
    "Ending condition "
    "(avg of last 5 cycles):"
)

print(
    f"  mean   "
    f"{het_df['end_s11_psia'].mean():.3f} psia"
)

print(
    f"  std    "
    f"{het_df['end_s11_psia'].std():.3f} psia"
)

print(
    f"  min    "
    f"{het_df['end_s11_psia'].min():.3f}"
)

print(
    f"  max    "
    f"{het_df['end_s11_psia'].max():.3f}"
)

print(
    f"  range  "
    f"{het_df['end_s11_psia'].max() - het_df['end_s11_psia'].min():.3f} "
    "psia\n"
)

print(
    "Degradation rate "
    "(slope, psia / cycle):"
)

print(
    f"  mean   "
    f"{het_df['slope_psia_per_cycle'].mean():.5f}"
)

print(
    f"  std    "
    f"{het_df['slope_psia_per_cycle'].std():.5f}"
)

print(
    f"  min    "
    f"{het_df['slope_psia_per_cycle'].min():.5f}"
)

print(
    f"  max    "
    f"{het_df['slope_psia_per_cycle'].max():.5f}"
)

fastest_over_slowest = (
    het_df["slope_psia_per_cycle"].max()
    / het_df["slope_psia_per_cycle"].min()
)

print(
    f"  ratio fastest/slowest = "
    f"{fastest_over_slowest:.1f}x\n"
)

print(
    "Lifetime "
    "(cycles to failure):"
)

print(
    f"  mean   "
    f"{het_df['lifetime'].mean():.0f}"
)

print(
    f"  std    "
    f"{het_df['lifetime'].std():.0f}"
)

print(
    f"  min    "
    f"{het_df['lifetime'].min()}"
)

print(
    f"  max    "
    f"{het_df['lifetime'].max()}"
)

print(
    f"  ratio longest/shortest = "
    f"{het_df['lifetime'].max() / het_df['lifetime'].min():.1f}x"
)

# --- Save the table ---

het_path = os.path.join(
    RESULTS_DIR,
    "fd001_engine_heterogeneity.csv"
)

het_df.to_csv(
    het_path,
    index=False
)

print(
    f"\nSaved heterogeneity table to:\n"
    f"{het_path}"
)

# ============================================================
# Part 2 — Visualisation
# All 100 engines overlaid in faint grey
# with the fleet mean curve on top
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 6)
)

# Plot all 100 engines in faint grey

for engine_id in train["unit"].unique():
    engine = train[
        train["unit"] == engine_id
    ]

    ax.plot(
        engine["cycle"],
        engine[SENSOR],
        color="grey",
        linewidth=0.5,
        alpha=0.20
    )

# Overlay the fleet mean at each cycle

cycle_max_for_mean = 250

fleet_mean = (
    train[
        train["cycle"] <= cycle_max_for_mean
    ]
    .groupby("cycle")[SENSOR]
    .mean()
)

ax.plot(
    fleet_mean.index,
    fleet_mean.values,
    color="#D9822B",
    linewidth=2.5,
    label=f"Fleet mean ({SENSOR})"
)

ax.set_xlabel(
    "Cycle"
)

ax.set_ylabel(
    f"{SENSOR} "
    "(Ps30 — static pressure at HPC outlet, psia)"
)

ax.legend(
    loc="upper left",
    fontsize=10
)

fig.tight_layout()

fig_path = os.path.join(
    FIGURES_DIR,
    f"fd001_heterogeneity_overlay_{SENSOR}.png"
)

fig.savefig(
    fig_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(
    f"\nSaved figure to:\n"
    f"{fig_path}"
)

# ============================================================
# Part 3 — Companion diagram
# Distributions of start, end, slope, and lifetime
# ============================================================

fig2, axes = plt.subplots(
    2,
    2,
    figsize=(14, 9)
)

# Starting-condition distribution

axes[0, 0].hist(
    het_df["start_s11_psia"],
    bins=20,
    color="#2E75B6",
    edgecolor="white"
)

axes[0, 0].set_xlabel(
    "Starting s11 "
    "(avg of first 5 cycles, psia)"
)

axes[0, 0].set_ylabel(
    "Number of engines"
)

# Ending-condition distribution

axes[0, 1].hist(
    het_df["end_s11_psia"],
    bins=20,
    color="#D9822B",
    edgecolor="white"
)

axes[0, 1].set_xlabel(
    "Ending s11 "
    "(avg of last 5 cycles, psia)"
)

axes[0, 1].set_ylabel(
    "Number of engines"
)

# Degradation-rate distribution

axes[1, 0].hist(
    het_df["slope_psia_per_cycle"] * 1000,
    bins=20,
    color="#1E8449",
    edgecolor="white"
)

axes[1, 0].set_xlabel(
    "Degradation rate "
    "(psia per 1000 cycles)"
)

axes[1, 0].set_ylabel(
    "Number of engines"
)

# Lifetime distribution

axes[1, 1].hist(
    het_df["lifetime"],
    bins=20,
    color="#5B4B8A",
    edgecolor="white"
)

axes[1, 1].set_xlabel(
    "Lifetime (cycles)"
)

axes[1, 1].set_ylabel(
    "Number of engines"
)

fig2.tight_layout()

fig2_path = os.path.join(
    FIGURES_DIR,
    "fd001_heterogeneity_distributions.png"
)

fig2.savefig(
    fig2_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(
    f"\nSaved distribution diagram to:\n"
    f"{fig2_path}"
)

In [ ]:
# ============================================================
# 01_EDA.ipynb
# Section 12 — Averaged Sensor Trends with Confidence Bands
# Purpose: enrich the Section 7 plot with ±1 std bands showing
#          engine-to-engine spread at each RUL value.
# Paths: uses FIGURES_DIR from Section 1.
# ============================================================

# --- Build the same temporary linear RUL column used in Section 7 ---
max_cycle_per_engine = train.groupby("unit")["cycle"].transform("max")
train_eda = train.copy()
train_eda["rul_linear"] = max_cycle_per_engine - train_eda["cycle"]

# --- The 21 sensors ---
sensor_cols = [f"s{i}" for i in range(1, 22)]

# --- Limit to the last 200 cycles before failure for visibility ---
LOOKBACK = 200
data = train_eda[train_eda["rul_linear"] <= LOOKBACK]

# --- Compute MEAN and STD per RUL value, per sensor ---
# Groupby RUL → aggregate each sensor with mean and std across engines at that RUL.
agg = data.groupby("rul_linear")[sensor_cols].agg(["mean", "std"])

# agg has hierarchical columns: (sensor, "mean") and (sensor, "std")
# Re-organize: get mean and std as separate dataframes for easier plotting.
means = agg.xs("mean", axis=1, level=1).sort_index(ascending=False)
stds  = agg.xs("std",  axis=1, level=1).sort_index(ascending=False)

# --- Plot: 7×3 grid of sensor subplots ---
fig, axes = plt.subplots(7, 3, figsize=(15, 18))
axes = axes.flatten()

for i, col in enumerate(sensor_cols):
    ax = axes[i]
    mean_curve = means[col]
    std_curve  = stds[col]

    # Mean line
    ax.plot(mean_curve.index, mean_curve.values,
            color="#2E75B6", linewidth=1.4, label="Fleet mean")

    # ±1 std band (shaded) — describes engine-to-engine spread
    ax.fill_between(mean_curve.index,
                    mean_curve.values - std_curve.values,
                    mean_curve.values + std_curve.values,
                    color="#2E75B6", alpha=0.20, label="±1 std (engine spread)")

    ax.set_title(col, fontsize=10)
    ax.set_xlabel("RUL (cycles to failure)", fontsize=8)
    ax.invert_xaxis()                                       # failure on the right
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=7, loc="best")


fig.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "fd001_avg_sensor_trends_with_bands.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure to:\n{fig_path}\n")

# ============================================================
# Companion: quantify how the band WIDTH changes from healthy to failure.
# If bands narrow toward failure -> engines converge to similar end states.
# If bands widen toward failure -> engines diverge near failure.
# ============================================================
print("=== Band width comparison: early life (RUL ≈ 200) vs. near failure (RUL ≈ 10) ===\n")
print(f"{'sensor':>6} | {'std at RUL=200':>16} | {'std at RUL=10':>14} | {'ratio (end/start)':>17}")
print("-" * 65)

for col in sensor_cols:
    # mean std over a small window of RUL values to avoid noise
    std_early = stds.loc[(stds.index >= 195) & (stds.index <= 200), col].mean()
    std_late  = stds.loc[(stds.index >= 5)   & (stds.index <= 15),  col].mean()
    if pd.isna(std_early) or std_early == 0:
        ratio_str = "—"
    else:
        ratio_str = f"{std_late / std_early:.2f}"
    print(f"{col:>6} | {std_early:>16.4f} | {std_late:>14.4f} | {ratio_str:>17}")

In [ ]:
# ============================================================
# Early-degradation analysis
#
# Question: do short- and long-lived engines differ
# from the first cycles?
#
# Design: predict total lifetime from the first N cycles only
# using five-fold cross-validation.
#
# Every engine has at least 128 cycles, so every tested window
# is available for all 100 engines.
# ============================================================

# --- Standalone setup ---

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)
RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)
FIGURES_DIR = os.path.join(
    PROJECT_ROOT,
    "figures"
)
SRC_DIR = os.path.join(
    PROJECT_ROOT,
    "src"
)

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from preprocessing import FINAL_FEATURES_FD001

train = pd.read_parquet(
    os.path.join(
        PROCESSED_DIR,
        "train_FD001_processed.parquet"
    )
)

print(
    f"train: {train.shape} | "
    f"sensors: {len(FINAL_FEATURES_FD001)}"
)

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr, mannwhitneyu

WINDOWS = [20, 30, 50, 80]

life = (
    train
    .groupby("unit")["cycle"]
    .max()
)


def early_features(df, n):
    """
    Calculate the mean, standard deviation, and slope
    of each sensor over the first n cycles.
    """

    rows = []

    for unit, engine_data in df.groupby("unit"):
        engine_data = (
            engine_data
            .sort_values("cycle")
            .head(n)
        )

        x = np.arange(len(engine_data))
        row = {"unit": unit}

        for sensor in FINAL_FEATURES_FD001:
            values = engine_data[sensor].values

            row[f"{sensor}_mean"] = values.mean()
            row[f"{sensor}_std"] = values.std()
            row[f"{sensor}_slope"] = np.polyfit(
                x,
                values,
                1
            )[0]

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# 1. Can early cycles predict total lifetime?
# ============================================================

y = life.values

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []
predictions = {}

print(
    "Predicting TOTAL lifetime from early cycles only "
    "(Ridge, 5-fold CV)"
)

print(
    f"{'window':>8}"
    f"{'R²':>9}"
    f"{'MAE':>8}"
    f"{'Spearman':>11}"
    f"{'p':>10}"
)

print("-" * 46)

for window in WINDOWS:
    X = (
        early_features(train, window)
        .drop(columns="unit")
        .values
    )

    model = make_pipeline(
        StandardScaler(),
        RidgeCV(
            alphas=np.logspace(
                -2,
                3,
                30
            )
        )
    )

    predicted = cross_val_predict(
        model,
        X,
        y,
        cv=cv
    )

    rho, p_value = spearmanr(
        predicted,
        y
    )

    predictions[window] = predicted

    results.append(
        {
            "early_window": window,
            "r2": round(
                r2_score(y, predicted),
                3
            ),
            "mae": round(
                mean_absolute_error(y, predicted),
                1
            ),
            "spearman": round(rho, 3),
            "p_value": round(p_value, 4)
        }
    )

    print(
        f"{window:>8}"
        f"{results[-1]['r2']:>9.3f}"
        f"{results[-1]['mae']:>8.1f}"
        f"{rho:>11.3f}"
        f"{p_value:>10.4f}"
    )

results_df = pd.DataFrame(results)

results_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "fd001_early_signal.csv"
    ),
    index=False
)

print(
    "\nBaseline (always predict the mean): "
    f"MAE = {np.mean(np.abs(y - y.mean())):.1f} cycles"
)

# ============================================================
# 2. Per-feature test
# Short-lived Q1 versus long-lived Q4 engines
# ============================================================

q1 = life.quantile(0.25)
q3 = life.quantile(0.75)

short = life[
    life <= q1
].index

long_ = life[
    life >= q3
].index

summary_50 = early_features(
    train,
    50
)

feature_columns = [
    column
    for column in summary_50.columns
    if column != "unit"
]

p_values = [
    (
        column,
        mannwhitneyu(
            summary_50[
                summary_50["unit"].isin(short)
            ][column],
            summary_50[
                summary_50["unit"].isin(long_)
            ][column]
        )[1]
    )
    for column in feature_columns
]

p_values.sort(
    key=lambda item: item[1]
)

bonferroni_threshold = (
    0.05
    / len(feature_columns)
)

n_significant = sum(
    1
    for _, p_value in p_values
    if p_value < bonferroni_threshold
)

print(
    f"\nShort (n={len(short)}) versus "
    f"long (n={len(long_)}) engines, "
    "first 50 cycles:"
)

print(
    "  features significant after Bonferroni: "
    f"{n_significant} / {len(feature_columns)}"
)

print(
    f"  strongest: {p_values[0][0]} "
    f"(p = {p_values[0][1]:.4f}, "
    f"threshold {bonferroni_threshold:.5f})"
)

top_sensor = (
    p_values[0][0]
    .rsplit("_", 1)[0]
)

# ============================================================
# 3. Figure
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4.5)
)

# Cross-validated R²

axes[0].plot(
    results_df["early_window"],
    results_df["r2"],
    marker="o",
    color="#2E75B6",
    linewidth=2
)

axes[0].axhline(
    0,
    color="grey",
    linestyle=":"
)

axes[0].set_xlabel(
    "Early window (cycles)"
)

axes[0].set_ylabel(
    "Cross-validated R²"
)

# Short-lived versus long-lived engines

for engine_ids, label, colour in [
    (
        short,
        "short-lived (Q1)",
        "#C55A11"
    ),
    (
        long_,
        "long-lived (Q4)",
        "#548235"
    )
]:
    subset = train[
        train["unit"].isin(engine_ids)
    ]

    grouped = (
        subset
        .groupby("cycle")[top_sensor]
        .agg(
            [
                "median",
                lambda values: values.quantile(0.25),
                lambda values: values.quantile(0.75)
            ]
        )
    )

    grouped.columns = [
        "median",
        "lower",
        "upper"
    ]

    grouped = grouped[
        grouped.index <= 150
    ]

    axes[1].plot(
        grouped.index,
        grouped["median"],
        color=colour,
        linewidth=2,
        label=label
    )

    axes[1].fill_between(
        grouped.index,
        grouped["lower"],
        grouped["upper"],
        color=colour,
        alpha=0.18
    )

axes[1].set_xlabel(
    "Cycle"
)

axes[1].set_ylabel(
    f"{top_sensor} (scaled)"
)

axes[1].legend()

# True versus predicted lifetime

best_window = WINDOWS[-1]

axes[2].scatter(
    y,
    predictions[best_window],
    color="#548235",
    s=35,
    alpha=0.75
)

limits = [
    y.min() - 10,
    y.max() + 10
]

axes[2].plot(
    limits,
    limits,
    "k--",
    linewidth=1
)

axes[2].set_xlabel(
    "True lifetime"
)

axes[2].set_ylabel(
    f"Predicted lifetime "
    f"(first {best_window} cycles)"
)

plt.tight_layout()

figure_path = os.path.join(
    FIGURES_DIR,
    "fd001_early_signal.png"
)

plt.savefig(
    figure_path,
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print(
    f"\nSaved: {figure_path}"
)

print(
    "Saved: fd001_early_signal.csv"
)

## EDA Summary — FD001

**Dataset facts (Section 2).** 100 training engines (run to failure),
100 test engines (truncated before failure), true RUL supplied separately.
Train shape (20631, 26), test shape (13096, 26). Single operating condition,
single fault mode (HPC degradation). Source: Saxena et al. 2008.

**Data quality (Section 3).** Zero missing values across train/test/y_test.
Zero duplicate rows, zero (unit, cycle) duplicates. All 26 columns numeric;
train/test dtypes match. Outlier inspection (1.5·IQR rule) shows flagged
values cluster at low RUL (near failure), confirming they are the
degradation signal, not noise.
**Decision: no outlier removal.** Evidence:
`results/fd001_outlier_report.csv`,
`figures/fd001_outliers_are_signal_<sensor>.png`.

**Constant / non-informative features (Section 3).** Seven sensors have
std ≈ 0: **s1, s5, s10, s16, s18, s19** (exact zero), and **s6**
(std = 0.001389, near-constant with rare drop events on individual
engines but no fleet-level degradation signal in the averaged-by-RUL plot).
Operational settings **op1 (std 0.002), op2 (std 0.0003), op3 (std 0)** are
effectively constant — consistent with FD001's single operating condition.
→ Drop all ten in preprocessing. Physical interpretation: these sensors
either measure quantities that depend on operating conditions that don't
vary in FD001 (e.g. T2, P2, farB, epr) or are commanded/demanded signals
not influenced by engine wear (Nf_dmd, PCNfR_dmd). See the Sensor Feature
Table in `thesis_notes` for details.
Evidence: `results/fd001_feature_std.csv`,
`figures/fd001_sensor_trajectories_engine1.png`.

**Redundant sensor pair (Section 5).** One pair exceeds |r| > 0.95:
**s9 ↔ s14 (r = 0.96)**. All other correlations remain below 0.95
(next-highest: s11↔s12 at −0.85). → Drop **s14**, keep s9.
Justification is both statistical AND physical: NRc = Nc / √θ where
θ = T₂/288.15K. In FD001 the inlet temperature T₂ is constant (single
operating condition), so NRc reduces to a constant multiple of Nc — the
two sensors carry the same information by definition. Source: standard
gas-turbine corrected-speed formula (Walsh & Fletcher *Gas Turbine
Performance*; HandWiki engineering reference; US Patent 9,512,784).
Evidence: `figures/fd001_correlation_heatmap.png`,
`results/fd001_high_correlation_pairs.csv`,
full justification in `thesis_notes` §3.

**Engine lifetimes (Section 6).** Range 128–362 cycles, mean 206,
median 199, std 46, IQR 177–229. **100 / 100 engines (100 %) exceed the
125-cycle cap** — the cap sits below the shortest-lived engine, so the
piecewise label flattens RUL only during the healthy phase of every
trajectory and never destroys real degradation information. The 125-cap
convention originates in Heimes 2008 (PHM 2008 Challenge, 2nd place) and
is adopted in essentially all later C-MAPSS literature. Evidence:
`figures/fd001_engine_lifetime_distribution.png`,
`results/fd001_engine_lifetimes.csv`.

**Degradation signal — fleet-level confirmation (Sections 4 & 7).**
Trending sensors confirmed on a single engine (Section 4) and across the
full fleet aligned by RUL, last 200 cycles, n = 100 (Section 7):
**rising toward failure** — s2, s3, s4, s8, s9, s11, s13, s14, s15, s17;
**falling toward failure** — s7, s12, s20, s21;
**flat** — s1, s5, s6, s10, s16, s18, s19.
Averaged trends are smooth and largely exponential (almost flat in early
life, sharp rise/fall in the last ~50 cycles), which is fleet-level visual
evidence that the chosen feature set carries a genuine degradation signal.
Evidence: `figures/fd001_sensor_trajectories_engine1.png`,
`figures/fd001_avg_sensor_trends_by_rul.png`.

**Degradation pattern — verified monotonic, no maintenance events
(Section 10).** Three complementary tests on s11 across 100 engines
(20-cycle smoothing to separate trend from sensor noise):
(A) Net rise (end − start): mean 0.678 psia, range 0.45–0.96 — **100/100
engines positive**. (B) Max drawdown from running peak: mean 0.046 psia,
max 0.110 — only ~7 % of the net rise, consistent with Gaussian sensor
noise rather than real recoveries. (C) Fraction of cycles with increasing
smoothed value: mean 0.661, all engines > 0.58. **Zero engines flagged.**
**Conclusion: FD001 contains no simulated maintenance events** — engines
degrade monotonically from healthy state to failure. Consistent with
Saxena 2008 §V: "the fault grows in magnitude until system failure."
Consequence for modeling: pure RUL regression is the appropriate framing;
no "time since last maintenance" feature is required.
Evidence: `figures/fd001_monotonicity_check_s11.png`,
`figures/fd001_monotonicity_metrics_s11.png`,
`results/fd001_monotonicity_s11_v2.csv`.

**Inter-engine heterogeneity (Section 11).** The 100 engines share the
same simulator model but are **not identical instances**. Across the fleet:
starting s11 ranges 47.06–47.66 psia (range 0.60 psia, std 0.15);
ending s11 ranges 47.94–48.27 psia (range 0.33 psia, std 0.05);
degradation rate ranges 0.0017–0.0057 psia/cycle, **fastest/slowest = 3.3×**;
lifetime ranges 128–362 cycles, longest/shortest = 2.8×.
Ending condition (on s11) is tighter than starting condition — engines
converge toward a similar failure threshold (consistent with the
simulator's cumulative health-index design, Saxena 2008 §IV) but start
from varied initial wear and degrade at varied rates.
**Consequence for modeling:** the same cycle number means different things
for different engines (cycle 100 could be near-healthy for a slow degrader
or near-failure for a fast one). Models must infer the rate from the recent
sensor trajectory, not from absolute time elapsed.
Evidence: `figures/fd001_heterogeneity_overlay_s11.png`,
`figures/fd001_heterogeneity_distributions.png`,
`results/fd001_engine_heterogeneity.csv`.

**Engine-spread bands on averaged trends (Section 12).** Averaged sensor
trends from Section 7 overlaid with ±1 standard deviation bands showing
engine-to-engine spread at each RUL value. Comparing band width at
RUL ≈ 200 (early life) vs. RUL ≈ 10 (near failure) reveals two patterns:

- **Most trending sensors CONVERGE near failure** (band-width ratio < 1):
  s11 (0.64), s4 (0.71), s7 (0.75), s12 (0.77), s15 (0.78), s20 (0.79),
  s17 (0.82), s2 (0.84), s3 (0.85), s21 (0.77). Engines reach similar
  late-life sensor states regardless of their early-life starting point —
  consistent with the simulator's cumulative-health-index design.
- **The core-speed sensors DIVERGE near failure** (band-width ratio ≫ 1):
  s9 (5.21×), s14 (4.78×). Engines reach failure at significantly different
  core-speed states. Physically: Nc (core spool speed) is sensitive to how
  HPC degradation distributes between compressor-side and turbine-side
  damage — engines that took different degradation paths arrive at failure
  with different Nc values, even when other sensors converge.
- **s8 and s13 stay roughly constant in spread** (ratio ≈ 1.06): minor
  trending sensors whose engine-to-engine variability is largely noise-driven.

**Consequence for modeling.** Two-part interpretation:
(a) **Late life is more predictable** for most sensors (convergence) —
this is part of why the piecewise-125 RUL label works: predicting precise
RUL during the noisy early life is harder than during the convergent late
phase.
(b) **The core-speed group (s9) carries diagnostic information** in its
late-life spread, not just its mean: which "failure mode" an engine
expresses shows up in s9 variability. This is information a sequence model
(LSTM, Chronos) can exploit via the temporal pattern around s9, while a
single-row regressor cannot.
Evidence: `figures/fd001_avg_sensor_trends_with_bands.png`.

### Decisions handed to 02_Preprocessing.ipynb

1. **Feature removal:** drop **op1, op2, op3** (near-constant, single operating
   condition) + **s1, s5, s6, s10, s16, s18, s19** (constant / no fleet-level
   signal) + **s14** (redundant with s9, r = 0.96 — physically justified).
   10 columns removed.
2. **Final feature set (13 sensors):** **s2, s3, s4, s7, s8, s9, s11, s12,
   s13, s15, s17, s20, s21**.
3. **RUL labels — both built as a planned ablation:**
   - `RUL_linear` = max_cycle − current_cycle (no cap)
   - `RUL_piecewise125` = same, clipped at 125
     (Heimes 2008 convention; justified by 100 % of engines exceeding the cap).
4. **Scaling:** Min-Max to [0, 1], **fit on TRAIN only** (no leakage),
   transform applied to both train and test with the same scaler.
5. **No outlier removal** (degradation tail = signal).
6. **No "time since maintenance" feature** (verified: no maintenance events in FD001).
7. **Models must read recent trajectory, not absolute cycle number**, because
   engines are heterogeneous in rate (3.3× spread). This is a methodological
   note for Notebook 07 interpretation.
8. **Test protocol (Phase 6):** predict RUL at each test engine's **last
   available cycle**, compare to `RUL_FD001.txt`. Identical protocol for
   all 4 models × 2 labels = 8 runs.

In [ ]:

import pandas as pd

# 26 columns: unit, cycle, 3 settings, 21 sensors
col_names = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

test_data = pd.read_csv(
    "{DATA_DIR}/../data/test_FD001.txt",
    sep=r"\s+", header=None, names=col_names
)

print(f"FD001 Test Data: {test_data.shape[0]} rows, {test_data.shape[1]} columns, "
      f"{test_data['unit'].nunique()} engines\n")

test_data

In [ ]:
# ============================================================
# NB01 — Figure for Section 5.7: start vs end distribution of s11
# Purpose: show the narrowing of between-engine spread from the
#          start of life to failure, without overlapping trajectories.
# Self-contained: reads the raw file directly (unscaled psia values).
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True)

COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)

N_EDGE = 5
rows = []
for unit, g in raw.groupby("unit"):
    g = g.sort_values("cycle")
    v = g["s11"].values
    rows.append({"unit": unit,
                 "initial": v[:N_EDGE].mean(),
                 "final":   v[-N_EDGE:].mean()})
het = pd.DataFrame(rows)

for col in ["initial", "final"]:
    s = het[col]
    print(f"{col:>8}: mean {s.mean():.3f} | std {s.std():.3f} | "
          f"min {s.min():.2f} | max {s.max():.2f} | range {s.max()-s.min():.2f}")

# --- Figure ---
fig, ax = plt.subplots(figsize=(7, 5.5))
data = [het["initial"].values, het["final"].values]
labels = ["Start of life\n(first 5 cycles)", "At failure\n(last 5 cycles)"]

bp = ax.boxplot(data, tick_labels=labels, widths=0.45, patch_artist=True,
                medianprops=dict(color="black", linewidth=1.6),
                flierprops=dict(marker="", linestyle="none"))
for patch, colour in zip(bp["boxes"], ["#BDD7EE", "#C5E0B4"]):
    patch.set_facecolor(colour)
    patch.set_edgecolor("black")

rng = np.random.default_rng(42)
for i, vals in enumerate(data, start=1):
    jitter = rng.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals,
               s=18, alpha=0.55, color="#333333", zorder=3, edgecolors="none")

# Headroom so annotations never touch the title
lo = min(v.min() for v in data)
hi = max(v.max() for v in data)
pad = (hi - lo) * 0.10
ax.set_ylim(lo - pad, hi + pad * 2.2)

# Annotations placed just above each box, inside the axes
for i, col in enumerate(["initial", "final"], start=1):
    rng_val = het[col].max() - het[col].min()
    ax.text(i, het[col].max() + pad * 0.35,
            f"range {rng_val:.2f} psia\nSD {het[col].std():.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="grey", alpha=0.85))

ax.set_ylabel("Sensor s11, static pressure at HPC outlet (psia)")

ax.grid(True, axis="y", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_s11_start_vs_end.png"),
            dpi=200, bbox_inches="tight")
plt.show()

het.to_csv(os.path.join(RESULTS_DIR, "fd001_s11_start_vs_end.csv"), index=False)
print("\nSaved: fd001_s11_start_vs_end.png + fd001_s11_start_vs_end.csv")

In [ ]:
# ============================================================
# NB01 — Figure for Section 5.7: start vs end distribution of s11
# Purpose: show the narrowing of between-engine spread from the
#          start of life to near failure.
# Self-contained: reads the raw file directly using unscaled psia values.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

COLS = (
    ["unit", "cycle", "op1", "op2", "op3"]
    + [f"s{i}" for i in range(1, 22)]
)

raw = pd.read_csv(
    RAW_PATH,
    sep=r"\s+",
    header=None,
    names=COLS
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

N_EDGE = 5

n_engines = raw["unit"].nunique()
cycle_counts = raw.groupby("unit").size()

assert n_engines == 100, (
    f"Expected 100 engines, but found {n_engines}."
)

assert cycle_counts.min() >= N_EDGE, (
    f"At least one engine has fewer than {N_EDGE} cycles."
)

# ------------------------------------------------------------
# Calculate initial and near-failure s11 levels
# ------------------------------------------------------------

rows = []

for unit, engine_data in raw.groupby("unit", sort=True):
    engine_data = engine_data.sort_values("cycle")
    s11_values = engine_data["s11"].to_numpy()

    rows.append(
        {
            "unit": unit,
            "initial": s11_values[:N_EDGE].mean(),
            "final": s11_values[-N_EDGE:].mean()
        }
    )

het = pd.DataFrame(rows)

assert len(het) == 100, (
    f"Expected 100 engine-level observations, but found {len(het)}."
)

# ------------------------------------------------------------
# Print descriptive statistics
# ------------------------------------------------------------

for column in ["initial", "final"]:
    values = het[column]
    value_range = values.max() - values.min()

    print(
        f"{column:>8}: "
        f"mean {values.mean():.3f} | "
        f"std {values.std():.3f} | "
        f"min {values.min():.2f} | "
        f"max {values.max():.2f} | "
        f"range {value_range:.2f}"
    )

# ------------------------------------------------------------
# Create figure
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5.5))

data = [
    het["initial"].to_numpy(),
    het["final"].to_numpy()
]

labels = [
    "Start of life\n(first 5 cycles)",
    "Near failure\n(last 5 cycles)"
]

bp = ax.boxplot(
    data,
    tick_labels=labels,
    widths=0.45,
    patch_artist=True,
    medianprops={
        "color": "black",
        "linewidth": 1.6
    },
    flierprops={
        "marker": "",
        "linestyle": "none"
    }
)

box_colours = ["#BDD7EE", "#C5E0B4"]

for patch, colour in zip(bp["boxes"], box_colours):
    patch.set_facecolor(colour)
    patch.set_edgecolor("black")

# Add one jittered point for each engine
random_generator = np.random.default_rng(42)

for position, values in enumerate(data, start=1):
    jitter = random_generator.uniform(
        -0.12,
        0.12,
        size=len(values)
    )

    ax.scatter(
        np.full(len(values), position) + jitter,
        values,
        s=18,
        alpha=0.55,
        color="#333333",
        edgecolors="none",
        zorder=3
    )

# ------------------------------------------------------------
# Set vertical limits with space for annotations
# ------------------------------------------------------------

lower_value = min(values.min() for values in data)
upper_value = max(values.max() for values in data)

padding = (upper_value - lower_value) * 0.10

ax.set_ylim(
    lower_value - padding,
    upper_value + padding * 1.6
)

# ------------------------------------------------------------
# Add range and standard-deviation annotations
# ------------------------------------------------------------

for position, column in enumerate(
    ["initial", "final"],
    start=1
):
    values = het[column]
    value_range = values.max() - values.min()

    ax.text(
        position,
        values.max() + padding * 0.35,
        (
            f"range {value_range:.2f} psia\n"
            f"SD {values.std():.2f}"
        ),
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        bbox={
            "boxstyle": "round,pad=0.3",
            "facecolor": "white",
            "edgecolor": "grey",
            "alpha": 0.85
        }
    )

# No title inside the figure
ax.set_ylabel(
    "Sensor s11, static pressure at HPC outlet (psia)"
)

ax.grid(
    True,
    axis="y",
    alpha=0.3
)

ax.set_axisbelow(True)

plt.tight_layout()

# ------------------------------------------------------------
# Save figure and results
# ------------------------------------------------------------

png_path = os.path.join(
    FIGURES_DIR,
    "fd001_s11_start_vs_end.png"
)

pdf_path = os.path.join(
    FIGURES_DIR,
    "fd001_s11_start_vs_end.pdf"
)

csv_path = os.path.join(
    RESULTS_DIR,
    "fd001_s11_start_vs_end.csv"
)

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

het.to_csv(
    csv_path,
    index=False
)

plt.show()

print("\nSaved:")
print(png_path)
print(pdf_path)
print(csv_path)

In [ ]:
# ============================================================
# NB01 — Section 5.9: when does a predictive signal become detectable?
# Question: how well can an engine's total lifetime be predicted
#           from its first N cycles alone?
# Design: pre-specified regular grid of window lengths, capped at
#         120 because the shortest training engine survives 128
#         cycles. No engine is excluded, so there is no survivorship
#         bias and every window is evaluated on the same 100 engines.
# Self-contained: reads the raw training file directly.
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s15","s17","s20","s21"]
WINDOWS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]

COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)

lifetimes = raw.groupby("unit")["cycle"].max()
print(f"{raw['unit'].nunique()} engines | shortest {lifetimes.min()} cycles "
      f"| longest {lifetimes.max()} | mean {lifetimes.mean():.1f}")
assert lifetimes.min() > max(WINDOWS), "A window exceeds the shortest trajectory"

def window_features(window):
    """Three summary statistics per sensor over the first `window` cycles."""
    X, y = [], []
    for unit, g in raw.groupby("unit"):
        g = g.sort_values("cycle")
        seg = g[SENSORS].values[:window]
        x = np.arange(window)
        feats = []
        for j in range(seg.shape[1]):
            c = seg[:, j]
            feats.extend([c.mean(), c.std(), np.polyfit(x, c, 1)[0]])
        X.append(feats)
        y.append(len(g))                      # total lifetime of this engine
    return np.array(X, np.float64), np.array(y, np.float64)

baseline_mean = baseline_median = None
rows = []
for w in WINDOWS:
    X, y = window_features(w)
    if baseline_mean is None:
        # Two constant predictors: the mean defines R2, the median
        # minimises absolute error and is the stricter benchmark.
        baseline_mean   = mean_absolute_error(y, np.full_like(y, y.mean()))
        baseline_median = mean_absolute_error(y, np.full_like(y, np.median(y)))
        print(f"\nBaseline, predicting the fleet mean:   MAE {baseline_mean:.1f} cycles")
        print(f"Baseline, predicting the fleet median: MAE {baseline_median:.1f} cycles\n")
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    pred = cross_val_predict(model, X, y, cv=KFold(5, shuffle=True, random_state=42))
    rho, p = spearmanr(y, pred)
    rows.append({"window": w,
                 "r2":   r2_score(y, pred),
                 "mae":  mean_absolute_error(y, pred),
                 "spearman": rho,
                 "p_value": p,
                 "n_features": X.shape[1],
                 "n_engines": len(y)})
    print(f"  window {w:>4}: R2 {rows[-1]['r2']:>7.3f} | MAE {rows[-1]['mae']:>5.1f} "
          f"| rho {rho:>6.3f} | p {p:.4g}")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(RESULTS_DIR, "fd001_early_signal_grid.csv"), index=False)

# --- Figure: R2 and MAE against window length. No titles; the report
#     supplies the caption. ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

ax1.plot(res["window"], res["r2"], marker="o", color="#2E75B6", linewidth=2)
ax1.axhline(0, color="grey", linestyle="--", linewidth=1)
ax1.set_xlabel("Window length (first N cycles)")
ax1.set_ylabel("Cross-validated R²")
ax1.grid(True, alpha=0.3); ax1.set_axisbelow(True)

ax2.plot(res["window"], res["mae"], marker="o", color="#C55A11", linewidth=2)
ax2.axhline(baseline_mean, color="grey", linestyle="--", linewidth=1.4)
ax2.annotate("predicting the fleet mean",
             xy=(res["window"].iloc[1], baseline_mean), xytext=(0, 7),
             textcoords="offset points", fontsize=9, color="grey")
ax2.set_xlabel("Window length (first N cycles)")
ax2.set_ylabel("Cross-validated MAE (cycles)")
ax2.grid(True, alpha=0.3); ax2.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_early_signal_grid.png"),
            dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: fd001_early_signal_grid.csv + fd001_early_signal_grid.png")

In [ ]:
# ============================================================
# NB01 — Section 5.9: where do IQR-flagged values of s8 occur?
# Reported as a PROPORTION within each remaining-life bin, because
# the number of available observations is not constant across
# remaining life: every engine contributes cycles close to failure,
# whereas only long-lived engines contribute at high remaining life.
# Self-contained: reads the raw training file directly.
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True)

SENSOR   = "s8"
BIN_SIZE = 10          # remaining-life bin width, in cycles
MIN_OBS  = 30          # minimum observations for a bin to be reported

COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)

# Linear remaining life per row (Equation 4.1)
raw["RUL"] = raw.groupby("unit")["cycle"].transform("max") - raw["cycle"]

# Conventional 1.5 x IQR criterion on the pooled distribution
q1, q3 = raw[SENSOR].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
raw["flagged"] = (raw[SENSOR] < lo) | (raw[SENSOR] > hi)

n_flag = int(raw["flagged"].sum())
print(f"{SENSOR}: Q1 {q1:.4f} | Q3 {q3:.4f} | IQR {iqr:.4f}")
print(f"Bounds: [{lo:.4f}, {hi:.4f}]")
print(f"Flagged: {n_flag} of {len(raw)} rows ({100*n_flag/len(raw):.2f}%)\n")

# Proportion flagged within each remaining-life bin
raw["bin"] = (raw["RUL"] // BIN_SIZE) * BIN_SIZE
grp = raw.groupby("bin").agg(
    n_obs=("flagged", "size"),
    n_flagged=("flagged", "sum"),
).reset_index()
grp["proportion"] = grp["n_flagged"] / grp["n_obs"]
grp = grp[grp["n_obs"] >= MIN_OBS]          # drop sparsely populated bins

print(grp.head(12).to_string(index=False))

# --- Figure: proportion flagged vs remaining life, with support ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})

ax1.bar(grp["bin"], grp["proportion"] * 100, width=BIN_SIZE * 0.85,
        color="#C55A11", edgecolor="black", linewidth=0.6)
ax1.axhline(100 * n_flag / len(raw), color="grey", linestyle="--", linewidth=1.4)
ax1.annotate("overall flagged rate", xy=(grp["bin"].max() * 0.55,
             100 * n_flag / len(raw)), xytext=(0, 6),
             textcoords="offset points", fontsize=9, color="grey")
ax1.set_ylabel(f"Flagged observations of {SENSOR} (%)")

ax1.grid(True, axis="y", alpha=0.3); ax1.set_axisbelow(True)

ax2.bar(grp["bin"], grp["n_obs"], width=BIN_SIZE * 0.85,
        color="#BDD7EE", edgecolor="black", linewidth=0.6)
ax2.set_xlabel("Remaining useful life (cycles)")
ax2.set_ylabel("Observations\navailable")
ax2.grid(True, axis="y", alpha=0.3); ax2.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_outlier_proportion_s8.png"),
            dpi=200, bbox_inches="tight")
plt.show()

grp.to_csv(os.path.join(RESULTS_DIR, "fd001_outlier_proportion_s8.csv"), index=False)
print(f"\nSaved: fd001_outlier_proportion_s8.png + fd001_outlier_proportion_s8.csv")

In [ ]:
# ============================================================
# NB01 — Non-linear degradation shape
# Question: does a single full-history slope describe the trajectory,
#           or does the rate of change itself change over an engine's life?
# Method: per engine, time is normalised to [0, 1] so that engines of
#         different lengths are comparable. The slope over the first
#         third is compared with the slope over the last third, and a
#         quadratic coefficient is fitted over the full trajectory.
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s15","s17","s20","s21"]
FOCUS   = "s11"

COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)

rows = []
for unit, g in raw.groupby("unit"):
    g = g.sort_values("cycle")
    n = len(g)
    t = np.linspace(0.0, 1.0, n)          # normalised lifetime
    third = n // 3
    rec = {"unit": unit, "lifetime": n}
    for s in SENSORS:
        v = g[s].values.astype(float)
        rec[f"{s}_slope_full"]  = np.polyfit(t, v, 1)[0]
        rec[f"{s}_slope_early"] = np.polyfit(t[:third],  v[:third],  1)[0]
        rec[f"{s}_slope_late"]  = np.polyfit(t[-third:], v[-third:], 1)[0]
        rec[f"{s}_quad"]        = np.polyfit(t, v, 2)[0]
    rows.append(rec)
shape = pd.DataFrame(rows)

# --- Summary across the 13 sensors ---
summary = []
for s in SENSORS:
    e, l, q = shape[f"{s}_slope_early"], shape[f"{s}_slope_late"], shape[f"{s}_quad"]
    rising = shape[f"{s}_slope_full"].mean() > 0
    # "steeper late" means the late slope moves further in the sensor's own direction
    steeper = (l > e).sum() if rising else (l < e).sum()
    summary.append({
        "sensor": s, "direction": "rising" if rising else "falling",
        "slope_early_mean": e.mean(), "slope_late_mean": l.mean(),
        "engines_steeper_late": int(steeper),
        "quad_mean": q.mean(),
        "engines_quad_same_sign_as_trend":
            int((q > 0).sum() if rising else (q < 0).sum()),
    })
summ = pd.DataFrame(summary)
print("Early-third vs late-third slope (normalised time), 100 engines\n")
print(summ.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

# --- Figure: two panels ---
fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: paired early -> late slope for the focus sensor
e = shape[f"{FOCUS}_slope_early"].values
l = shape[f"{FOCUS}_slope_late"].values
for i in range(len(e)):
    axA.plot([0, 1], [e[i], l[i]], color="#999999", alpha=0.35, linewidth=0.8, zorder=1)
axA.scatter(np.zeros_like(e), e, s=22, color="#2E75B6", zorder=3, label="First third")
axA.scatter(np.ones_like(l),  l, s=22, color="#C55A11", zorder=3, label="Last third")
axA.set_xticks([0, 1]); axA.set_xticklabels(["First third", "Last third"])
axA.set_xlim(-0.25, 1.25)
axA.set_ylabel(f"Slope of {FOCUS} (psia per normalised lifetime)")
axA.set_title(f"Degradation rate of {FOCUS} increases within each engine",
              fontweight="bold", fontsize=11)
axA.grid(True, axis="y", alpha=0.3); axA.set_axisbelow(True)

# Panel B: how many engines are steeper late, per sensor
order = summ.sort_values("engines_steeper_late")
colours = ["#548235" if v == 100 else "#C55A11" for v in order["engines_steeper_late"]]
axB.barh(order["sensor"], order["engines_steeper_late"], color=colours,
         edgecolor="black", linewidth=0.6)
axB.axvline(50, color="grey", linestyle="--", linewidth=1.2)
axB.set_xlabel("Engines with a steeper late-third slope (of 100)")
axB.set_xlim(0, 105)
axB.set_title("Acceleration is present across the retained sensors",
              fontweight="bold", fontsize=11)
axB.grid(True, axis="x", alpha=0.3); axB.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_degradation_shape.png"),
            dpi=300, bbox_inches="tight")
plt.show()

shape.to_csv(os.path.join(RESULTS_DIR, "fd001_degradation_shape_per_engine.csv"), index=False)
summ.to_csv(os.path.join(RESULTS_DIR, "fd001_degradation_shape_summary.csv"), index=False)
print("\nSaved: fd001_degradation_shape.png + two CSVs")

In [ ]:
# ============================================================
# NB01 — Between-engine variation across the retained sensors
# Question: do engines converge towards failure in every sensor,
#           or only in some?
# Method: per engine and sensor, the initial level is the mean of the
#         first five cycles and the final level the mean of the last
#         five. The ratio of the between-engine SD at the end to that
#         at the start summarises how the spread of the fleet changes
#         over an engine's life. Three descriptive bands are used,
#         fixed before inspecting the results:
#           below 0.80   -> fleet converges
#           0.80 to 1.25 -> spread largely unchanged
#           above 1.25   -> fleet diverges
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s15","s17","s20","s21"]
N_EDGE = 5
BAND_LOW, BAND_HIGH = 0.80, 1.25

# one hue, three steps: lightest = narrowing spread, darkest = widening
C_BELOW, C_WITHIN, C_ABOVE = "#9ECAE1", "#4292C6", "#08519C"

COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)

rows = []
for unit, g in raw.groupby("unit"):
    g = g.sort_values("cycle")
    rec = {"unit": unit}
    for s in SENSORS:
        v = g[s].values.astype(float)
        rec[f"{s}_initial"] = v[:N_EDGE].mean()
        rec[f"{s}_final"]   = v[-N_EDGE:].mean()
    rows.append(rec)
het = pd.DataFrame(rows)

def classify(r):
    if r < BAND_LOW:
        return "converging"
    elif r <= BAND_HIGH:
        return "largely unchanged"
    return "diverging"

summary = []
for s in SENSORS:
    i, f = het[f"{s}_initial"], het[f"{s}_final"]
    ratio = f.std() / i.std()
    summary.append({
        "sensor": s,
        "initial_mean": i.mean(), "initial_sd": i.std(),
        "initial_range": i.max() - i.min(),
        "final_mean":   f.mean(), "final_sd":   f.std(),
        "final_range":  f.max() - f.min(),
        "mean_change":  f.mean() - i.mean(),
        "sd_ratio":     ratio,
        "band":         classify(ratio),
    })
summ = pd.DataFrame(summary)
print("Between-engine spread at start and end of life, 100 engines\n")
print(summ.to_string(index=False, float_format=lambda x: f"{x:10.4f}"))
print("\nBand counts:")
print(summ["band"].value_counts().to_string())

# ---------------- Figure: SD ratio per sensor ----------------
def band_colour(r):
    if r < BAND_LOW:
        return C_BELOW
    elif r <= BAND_HIGH:
        return C_WITHIN
    return C_ABOVE

order   = summ.sort_values("sd_ratio")
colours = [band_colour(r) for r in order["sd_ratio"]]

fig, ax = plt.subplots(figsize=(6.3, 3.6), dpi=300)

ax.barh(order["sensor"], order["sd_ratio"], color=colours,
        edgecolor="#33333340", linewidth=0.4, height=0.72)

ax.axvline(BAND_LOW,  color="#666666", linestyle=":", linewidth=0.9)
ax.axvline(BAND_HIGH, color="#666666", linestyle=":", linewidth=0.9)

for i, r in enumerate(order["sd_ratio"]):
    ax.text(r + 0.09, i, f"{r:.2f}", va="center", ha="left",
            fontsize=7.5, color="#222222")

ax.set_xlim(0, order["sd_ratio"].max() * 1.15)
ax.set_xlabel("Between-engine SD at failure \u00f7 SD at start of life", fontsize=8)
ax.tick_params(axis="both", labelsize=8)

for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.spines["left"].set_linewidth(0.6)
ax.spines["bottom"].set_linewidth(0.6)

ax.legend(handles=[
        Patch(facecolor=C_BELOW,  label="Below 0.80 \u2014 spread narrows"),
        Patch(facecolor=C_WITHIN, label="0.80 to 1.25 \u2014 spread largely unchanged"),
        Patch(facecolor=C_ABOVE,  label="Above 1.25 \u2014 spread widens")],
    loc="lower right", fontsize=7.5, frameon=False,
    handlelength=1.1, handleheight=1.0, labelspacing=0.45)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_heterogeneity_all_sensors.png"),
            dpi=300, bbox_inches="tight")
plt.show()

het.to_csv(os.path.join(RESULTS_DIR, "fd001_heterogeneity_per_engine.csv"), index=False)
summ.to_csv(os.path.join(RESULTS_DIR, "fd001_heterogeneity_summary.csv"), index=False)
print("\nSaved: fd001_heterogeneity_all_sensors.png + two CSVs")

In [ ]:
# ============================================================
# NB01 — Between-engine variation across the retained sensors
# Question: do engines converge towards failure in every sensor,
#           or only in some?
# Method: per engine and sensor, the initial level is the mean of the
#         first five cycles and the final level the mean of the last
#         five. The ratio of the between-engine SD at the end to that
#         at the start summarises how the spread of the fleet changes
#         over an engine's life. Three descriptive bands are used,
#         fixed before inspecting the results:
#           below 0.80   -> fleet converges
#           0.80 to 1.25 -> spread largely unchanged
#           above 1.25   -> fleet diverges
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s15","s17","s20","s21"]
FOCUS, N_EDGE = "s11", 5
BAND_LOW, BAND_HIGH = 0.80, 1.25

COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)

rows = []
for unit, g in raw.groupby("unit"):
    g = g.sort_values("cycle")
    rec = {"unit": unit}
    for s in SENSORS:
        v = g[s].values.astype(float)
        rec[f"{s}_initial"] = v[:N_EDGE].mean()
        rec[f"{s}_final"]   = v[-N_EDGE:].mean()
    rows.append(rec)
het = pd.DataFrame(rows)

def classify(r):
    if r < BAND_LOW:
        return "converging"
    elif r <= BAND_HIGH:
        return "largely unchanged"
    return "diverging"

summary = []
for s in SENSORS:
    i, f = het[f"{s}_initial"], het[f"{s}_final"]
    ratio = f.std() / i.std()
    summary.append({
        "sensor": s,
        "initial_mean": i.mean(), "initial_sd": i.std(),
        "initial_range": i.max() - i.min(),
        "final_mean":   f.mean(), "final_sd":   f.std(),
        "final_range":  f.max() - f.min(),
        "mean_change":  f.mean() - i.mean(),
        "sd_ratio":     ratio,
        "band":         classify(ratio),
    })
summ = pd.DataFrame(summary)
print("Between-engine spread at start and end of life, 100 engines\n")
print(summ.to_string(index=False, float_format=lambda x: f"{x:10.4f}"))
print("\nBand counts:")
print(summ["band"].value_counts().to_string())

# --- Figure: two panels ---
fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 5.2))

# Panel A: focus sensor, start vs end
data = [het[f"{FOCUS}_initial"].values, het[f"{FOCUS}_final"].values]
bp = axA.boxplot(data, tick_labels=["Start of life\n(first 5 cycles)",
                                    "At failure\n(last 5 cycles)"],
                 widths=0.45, patch_artist=True,
                 medianprops=dict(color="black", linewidth=1.6),
                 flierprops=dict(marker="", linestyle="none"))
for patch, colour in zip(bp["boxes"], ["#BDD7EE", "#C5E0B4"]):
    patch.set_facecolor(colour); patch.set_edgecolor("black")
rng = np.random.default_rng(42)
for i, vals in enumerate(data, start=1):
    axA.scatter(np.full(len(vals), i) + rng.uniform(-0.12, 0.12, len(vals)),
                vals, s=16, alpha=0.5, color="#333333", zorder=3, edgecolors="none")
axA.set_ylabel(f"{FOCUS}, static pressure at HPC outlet (psia)")
axA.set_title(f"{FOCUS}: engines converge towards failure",
              fontweight="bold", fontsize=11)
axA.grid(True, axis="y", alpha=0.3); axA.set_axisbelow(True)

# Panel B: SD ratio for every retained sensor, coloured by band
def band_colour(r):
    if r < BAND_LOW:
        return "#548235"      # converging
    elif r <= BAND_HIGH:
        return "#999999"      # largely unchanged
    return "#C00000"          # diverging

order = summ.sort_values("sd_ratio")
colours = [band_colour(r) for r in order["sd_ratio"]]
axB.barh(order["sensor"], order["sd_ratio"], color=colours,
         edgecolor="black", linewidth=0.6)
axB.axvline(BAND_LOW,  color="grey", linestyle=":", linewidth=1.2)
axB.axvline(BAND_HIGH, color="grey", linestyle=":", linewidth=1.2)

# Value labels, since s9 stretches the axis and compresses the rest
for i, r in enumerate(order["sd_ratio"]):
    axB.text(r + 0.12, i, f"{r:.2f}", va="center", fontsize=8)
axB.set_xlim(0, order["sd_ratio"].max() * 1.18)

axB.set_xlabel("Between-engine SD at failure ÷ SD at start of life")
axB.set_title("Convergence is not universal across sensors",
              fontweight="bold", fontsize=11)
axB.grid(True, axis="x", alpha=0.3); axB.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_heterogeneity_all_sensors.png"),
            dpi=300, bbox_inches="tight")
plt.show()

het.to_csv(os.path.join(RESULTS_DIR, "fd001_heterogeneity_per_engine.csv"), index=False)
summ.to_csv(os.path.join(RESULTS_DIR, "fd001_heterogeneity_summary.csv"), index=False)
print("\nSaved: fd001_heterogeneity_all_sensors.png + two CSVs")

In [ ]:
# ============================================================
# NB01 — Where do IQR-flagged values occur, across all retained sensors?
# Method: the 1.5 x IQR criterion is applied to each sensor's pooled
#         training distribution. Flagged observations are then reported
#         as a proportion of the observations available within each
#         remaining-life bin. Bins are capped at 119 because every
#         engine survives at least 128 cycles, so all bins below that
#         contain the same number of observations.
# ============================================================
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS  = ["s2","s3","s4","s7","s8","s9","s11","s12","s13","s15","s17","s20","s21"]
BIN_SIZE, MAX_RUL = 10, 120

COLS = ["unit","cycle","op1","op2","op3"] + [f"s{i}" for i in range(1,22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)
raw["RUL"] = raw.groupby("unit")["cycle"].transform("max") - raw["cycle"]

sub = raw[raw["RUL"] < MAX_RUL].copy()
sub["bin"] = (sub["RUL"] // BIN_SIZE) * BIN_SIZE
support = sub.groupby("bin").size()
print("Observations per remaining-life bin:")
print(support.to_string(), "\n")

mat, overall = [], []
for s in SENSORS:
    q1, q3 = raw[s].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged_all = ((raw[s] < lo) | (raw[s] > hi)).mean()
    overall.append({"sensor": s, "overall_flagged_pct": 100 * flagged_all})
    sub["flag"] = (sub[s] < lo) | (sub[s] > hi)
    mat.append(sub.groupby("bin")["flag"].mean().reindex(support.index).values * 100)

mat = np.array(mat)
bins = support.index.values
prop = pd.DataFrame(mat, index=SENSORS,
                    columns=[f"{b}-{b+BIN_SIZE-1}" for b in bins])
print("Percentage of observations flagged, by sensor and remaining-life bin:\n")
print(prop.to_string(float_format=lambda x: f"{x:6.2f}"))

# --- Figure: heatmap with power-law colour scaling ---
# Trim the trailing bins in which no sensor is ever flagged
keep = mat.max(axis=0) > 0
mat_p, bins_p = mat[:, keep], bins[keep]
cols_p = [f"{b}-{b+BIN_SIZE-1}" for b in bins_p]

fig, ax = plt.subplots(figsize=(9, 5.5))
# Square-root scaling: without it the single high row compresses all
# other sensors into the lowest part of the colour range.
im = ax.imshow(mat_p, aspect="auto", cmap="OrRd",
               norm=PowerNorm(gamma=0.45, vmin=0, vmax=mat_p.max()))
ax.set_xticks(range(len(bins_p))); ax.set_xticklabels(cols_p, rotation=45, ha="right")
ax.set_yticks(range(len(SENSORS))); ax.set_yticklabels(SENSORS)
ax.set_xlabel("Remaining useful life (cycles)")
ax.set_ylabel("Sensor")
ax.set_title("IQR-flagged observations concentrate close to failure",
             fontweight="bold", fontsize=12, pad=10)

for i in range(mat_p.shape[0]):
    for j in range(mat_p.shape[1]):
        v = mat_p[i, j]
        if v >= 0.5:
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 20 else "black")

cbar = fig.colorbar(im, ax=ax, pad=0.02, ticks=[0, 1, 5, 10, 20, 30, 40])
cbar.set_label("Observations flagged (%)")
cbar.ax.set_yticklabels(["0", "1", "5", "10", "20", "30", "40"])

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "fd001_outlier_heatmap_all_sensors.png"),
            dpi=300, bbox_inches="tight")
plt.show()

prop.to_csv(os.path.join(RESULTS_DIR, "fd001_outlier_proportion_all_sensors.csv"))
pd.DataFrame(overall).to_csv(
    os.path.join(RESULTS_DIR, "fd001_outlier_overall_rate.csv"), index=False)
print("\nSaved: fd001_outlier_heatmap_all_sensors.png + two CSVs")

In [ ]:
# ============================================================
# NB01 — Where do IQR-flagged values occur across all
# retained sensors?
#
# Method:
# The 1.5 × IQR criterion is applied separately to each
# sensor's pooled training distribution.
#
# Flagged observations are reported as a proportion of the
# observations available within each remaining-life bin.
#
# RUL is restricted to 0–119 cycles. Since every FD001
# training engine survives for at least 128 cycles, every
# displayed 10-cycle bin contains observations from all
# 100 engines.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm


# ------------------------------------------------------------
# Paths and configuration
# ------------------------------------------------------------

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "train_FD001.txt"
)
FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS = [
    "s2", "s3", "s4", "s7", "s8", "s9", "s11",
    "s12", "s13", "s15", "s17", "s20", "s21"
]

BIN_SIZE = 10
MAX_RUL = 120

EXPECTED_BINS = np.arange(0, MAX_RUL, BIN_SIZE)

COLS = (
    ["unit", "cycle", "op1", "op2", "op3"]
    + [f"s{i}" for i in range(1, 22)]
)


# ------------------------------------------------------------
# Load the training data and calculate linear RUL
# ------------------------------------------------------------

raw = pd.read_csv(
    RAW_PATH,
    sep=r"\s+",
    header=None,
    names=COLS
)

raw["RUL"] = (
    raw.groupby("unit")["cycle"].transform("max")
    - raw["cycle"]
)


# ------------------------------------------------------------
# Restrict the analysis to RUL 0–119
# ------------------------------------------------------------

sub = raw.loc[raw["RUL"] < MAX_RUL].copy()

sub["bin"] = (
    sub["RUL"].floordiv(BIN_SIZE)
    .mul(BIN_SIZE)
    .astype(int)
)

support = (
    sub.groupby("bin")
    .size()
    .reindex(EXPECTED_BINS)
)

# Confirm that every displayed bin has equal support
assert support.notna().all(), (
    "At least one expected RUL bin is missing."
)

assert support.nunique() == 1, (
    "The displayed RUL bins do not have equal observation counts."
)

print("Observations per remaining-life bin:")
print(support.to_string())
print()


# ------------------------------------------------------------
# Apply the pooled 1.5 × IQR rule to every retained sensor
# ------------------------------------------------------------

proportion_rows = []
threshold_rows = []

for sensor in SENSORS:

    q1 = raw[sensor].quantile(0.25)
    q3 = raw[sensor].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    flagged_all = (
        (raw[sensor] < lower_bound)
        | (raw[sensor] > upper_bound)
    )

    flagged_sub = flagged_all.loc[sub.index]

    bin_proportions = (
        pd.DataFrame({
            "bin": sub["bin"],
            "flagged": flagged_sub
        })
        .groupby("bin")["flagged"]
        .mean()
        .reindex(EXPECTED_BINS)
        .mul(100)
    )

    proportion_rows.append(bin_proportions.to_numpy())

    threshold_rows.append({
        "sensor": sensor,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "overall_flagged_pct": 100 * flagged_all.mean()
    })


# ------------------------------------------------------------
# Create result tables
# ------------------------------------------------------------

mat = np.asarray(proportion_rows)

bin_labels = [
    f"{start}-{start + BIN_SIZE - 1}"
    for start in EXPECTED_BINS
]

prop = pd.DataFrame(
    mat,
    index=SENSORS,
    columns=bin_labels
)

prop.index.name = "sensor"

thresholds = pd.DataFrame(threshold_rows)

print(
    "Percentage of observations flagged, "
    "by sensor and remaining-life bin:\n"
)

print(
    prop.to_string(
        float_format=lambda value: f"{value:6.2f}"
    )
)


# ------------------------------------------------------------
# Keep bins contiguous
#
# Only trailing all-zero columns are removed. Internal zero
# columns remain, preserving the regular RUL axis.
# ------------------------------------------------------------

column_maximum = np.nanmax(mat, axis=0)
nonzero_columns = np.flatnonzero(column_maximum > 0)

if nonzero_columns.size:
    last_column = nonzero_columns[-1] + 1
else:
    last_column = mat.shape[1]

mat_plot = mat[:, :last_column]
bins_plot = EXPECTED_BINS[:last_column]

labels_plot = [
    f"{start}-{start + BIN_SIZE - 1}"
    for start in bins_plot
]


# ------------------------------------------------------------
# Create the heatmap
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5.6))

maximum_value = max(
    float(np.nanmax(mat_plot)),
    1.0
)

# Power-law scaling makes smaller proportions visible while
# retaining the full range of the largest proportions.
normalisation = PowerNorm(
    gamma=0.45,
    vmin=0,
    vmax=maximum_value
)

image = ax.imshow(
    mat_plot,
    aspect="auto",
    cmap="OrRd",
    norm=normalisation,
    interpolation="nearest"
)

ax.set_xticks(np.arange(len(labels_plot)))
ax.set_xticklabels(
    labels_plot,
    rotation=45,
    ha="right"
)

ax.set_yticks(np.arange(len(SENSORS)))
ax.set_yticklabels(SENSORS)

ax.set_xlabel("Remaining useful life (cycles)")
ax.set_ylabel("Sensor")

# No internal title. Use the numbered caption in the thesis.


# ------------------------------------------------------------
# Annotate values of at least 1%
# ------------------------------------------------------------

for row_index in range(mat_plot.shape[0]):
    for column_index in range(mat_plot.shape[1]):

        value = mat_plot[row_index, column_index]

        if np.isfinite(value) and value >= 1.0:

            text_colour = (
                "white"
                if image.norm(value) > 0.55
                else "black"
            )

            ax.text(
                column_index,
                row_index,
                f"{value:.0f}",
                ha="center",
                va="center",
                fontsize=8,
                color=text_colour
            )


# ------------------------------------------------------------
# Colour bar
# ------------------------------------------------------------

candidate_ticks = [0, 1, 5, 10, 20, 30, 40]
colourbar_ticks = [
    tick
    for tick in candidate_ticks
    if tick <= maximum_value
]

colourbar = fig.colorbar(
    image,
    ax=ax,
    pad=0.02,
    ticks=colourbar_ticks
)

colourbar.set_label("Observations flagged (%)")


# ------------------------------------------------------------
# Save and display
# ------------------------------------------------------------

plt.tight_layout()

figure_path = os.path.join(
    FIGURES_DIR,
    "fd001_outlier_heatmap_all_sensors.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
plt.close(fig)


# ------------------------------------------------------------
# Save numerical results
# ------------------------------------------------------------

proportion_path = os.path.join(
    RESULTS_DIR,
    "fd001_outlier_proportion_all_sensors.csv"
)

threshold_path = os.path.join(
    RESULTS_DIR,
    "fd001_outlier_thresholds_and_overall_rates.csv"
)

prop.to_csv(proportion_path)

thresholds.to_csv(
    threshold_path,
    index=False
)

print("\nSaved:")
print("• fd001_outlier_heatmap_all_sensors.png")
print("• fd001_outlier_proportion_all_sensors.csv")
print("• fd001_outlier_thresholds_and_overall_rates.csv")

In [ ]:
# ============================================================
# NB01 — Per-engine trend consistency (Figure 5.3)
# For each engine separately, the Spearman rank correlation between
# each sensor and the cycle index is computed. This measures the
# strength of a monotonic association over a whole trajectory; it
# does not imply that the sensor moves in one direction at every step.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
RAW_PATH     = os.path.join(PROJECT_ROOT, "data", "train_FD001.txt")
FIGURES_DIR  = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS = ["s2", "s3", "s4", "s7", "s8", "s9",
           "s11", "s12", "s13", "s15", "s17", "s20", "s21"]

COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
raw = pd.read_csv(RAW_PATH, sep=r"\s+", header=None, names=COLS)


# ------------------------------------------------------------
# Compute the summary table that the figure is built from
# ------------------------------------------------------------

rows = []
for sensor in SENSORS:
    rhos = np.array([
        spearmanr(g["cycle"], g[sensor]).statistic
        for _, g in raw.groupby("unit")
    ])
    dominant = 1 if np.median(rhos) > 0 else -1
    rows.append({
        "sensor": sensor,
        "median_rho": np.median(rhos),
        "q1_rho": np.percentile(rhos, 25),
        "q3_rho": np.percentile(rhos, 75),
        "direction_consistency_pct": 100 * np.mean(np.sign(rhos) == dominant),
        "direction": "increasing" if dominant > 0 else "decreasing",
        "n_engines": len(rhos),
    })

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RESULTS_DIR, "fd001_trend_consistency.csv"),
               index=False)

print("Per-engine trend consistency across 100 training engines\n")
print(summary.to_string(index=False, float_format=lambda x: f"{x:8.3f}"))

# Values quoted in Section 5.5
consistent = summary[summary["direction_consistency_pct"] == 100]
print(f"\nSensors with 100% direction agreement: {len(consistent)} of {len(summary)}")
print(f"Median |rho| among those: {consistent['median_rho'].abs().min():.3f} "
      f"to {consistent['median_rho'].abs().max():.3f}")
odd = summary[summary["direction_consistency_pct"] < 100]
if len(odd):
    print("\nSensors below 100% agreement:")
    print(odd[["sensor", "median_rho", "q1_rho", "q3_rho",
               "direction_consistency_pct"]].to_string(index=False))


# ------------------------------------------------------------
# Arrange sensors by median correlation
# ------------------------------------------------------------

plot_summary = (
    summary
    .sort_values("median_rho", ascending=False)
    .reset_index(drop=True)
)

y_positions = np.arange(len(plot_summary))

positive_colour = "#D97732"   # Increasing
negative_colour = "#5B7DBB"   # Decreasing

colours = [
    positive_colour if value >= 0 else negative_colour
    for value in plot_summary["median_rho"]
]


# ------------------------------------------------------------
# Create clean two-panel figure
# ------------------------------------------------------------

fig, (ax_left, ax_right) = plt.subplots(
    1, 2, figsize=(11, 7), sharey=True,
    gridspec_kw={"width_ratios": [1.8, 1], "wspace": 0.08}
)


# ------------------------------------------------------------
# Left panel: median Spearman correlation and interquartile range
# ------------------------------------------------------------

for i, row in plot_summary.iterrows():
    colour = colours[i]

    ax_left.hlines(y=i, xmin=row["q1_rho"], xmax=row["q3_rho"],
                   color=colour, linewidth=5, alpha=0.45, zorder=2)

    ax_left.vlines([row["q1_rho"], row["q3_rho"]],
                   ymin=i - 0.12, ymax=i + 0.12,
                   color=colour, linewidth=1.5, zorder=3)

    ax_left.scatter(row["median_rho"], i, s=65, color=colour,
                    edgecolor="white", linewidth=0.9, zorder=4)

ax_left.axvline(0, color="#333333", linestyle="--", linewidth=1)
ax_left.set_xlim(-1.05, 1.05)
ax_left.set_yticks(y_positions)
ax_left.set_yticklabels(plot_summary["sensor"], fontsize=10)
ax_left.set_xlabel("Median Spearman correlation with cycle\n"
                   "(horizontal line shows the interquartile range)")
ax_left.set_ylabel("Sensor")
ax_left.grid(axis="x", color="#D9D9D9", linewidth=0.7, alpha=0.7)


# ------------------------------------------------------------
# Right panel: percentage agreeing with the dominant direction
# ------------------------------------------------------------

agreement = plot_summary["direction_consistency_pct"].to_numpy()

ax_right.scatter(agreement, y_positions, s=65, color=colours,
                 edgecolor="white", linewidth=0.9, zorder=3)

for i, value in enumerate(agreement):
    if value >= 97:
        x_text, alignment = value - 1.2, "right"
    else:
        x_text, alignment = value + 1.2, "left"
    ax_right.text(x_text, i, f"{value:.0f}%", va="center", ha=alignment,
                  fontsize=9, color="#333333")

ax_right.set_xlim(50, 103)
ax_right.set_xticks([50, 60, 70, 80, 90, 100])
ax_right.set_xlabel("Engines agreeing with\nthe dominant direction")
ax_right.tick_params(axis="y", left=False, labelleft=False)
ax_right.grid(axis="x", color="#D9D9D9", linewidth=0.7, alpha=0.7)


# ------------------------------------------------------------
# Horizontal guides connecting both panels
# ------------------------------------------------------------

for axis in [ax_left, ax_right]:
    for y in y_positions:
        axis.axhline(y, color="#EEEEEE", linewidth=0.6, zorder=0)

ax_left.invert_yaxis()


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

legend_items = [
    Line2D([0], [0], marker="o", color="none",
           markerfacecolor=positive_colour, markeredgecolor="white",
           markersize=8, label="Increasing with cycle"),
    Line2D([0], [0], marker="o", color="none",
           markerfacecolor=negative_colour, markeredgecolor="white",
           markersize=8, label="Decreasing with cycle"),
]

fig.legend(handles=legend_items, loc="upper center",
           bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=False)

plt.tight_layout(rect=[0, 0, 1, 0.95])

figure_path = os.path.join(FIGURES_DIR,
                           "fd001_sensor_trend_consistency_clean.png")
fig.savefig(figure_path, dpi=300, bbox_inches="tight", facecolor="white")

plt.show()
plt.close(fig)

print(f"\nSaved: {figure_path}")

In [ ]:
# ============================================================
# FD001 — Sensor trajectories across normalized engine life
#
# Each cell shows the median standardized change from the
# engine-specific starting level across the 100 engines.
#
# Red  = increase from the starting level
# Blue = decrease from the starting level
# White = little or no change
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


# ------------------------------------------------------------
# Paths and configuration
# ------------------------------------------------------------

PROJECT_ROOT = os.path.join(DATA_DIR, "..")

RAW_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "train_FD001.txt"
)

FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SENSORS = [
    "s2", "s3", "s4", "s7", "s8", "s9", "s11",
    "s12", "s13", "s15", "s17", "s20", "s21"
]

COLS = (
    ["unit", "cycle", "op1", "op2", "op3"]
    + [f"s{i}" for i in range(1, 22)]
)

NUMBER_OF_BINS = 10
STARTING_CYCLES = 5


# ------------------------------------------------------------
# Load FD001 training data
# ------------------------------------------------------------

raw = pd.read_csv(
    RAW_PATH,
    sep=r"\s+",
    header=None,
    names=COLS
)

assert raw["unit"].nunique() == 100

raw = raw.sort_values(
    ["unit", "cycle"]
).reset_index(drop=True)


# ------------------------------------------------------------
# Convert cycle to relative engine life
#
# 0%   = beginning of the engine record
# 100% = failure cycle
# ------------------------------------------------------------

maximum_cycle = raw.groupby("unit")["cycle"].transform("max")

raw["relative_life"] = (
    (raw["cycle"] - 1)
    / (maximum_cycle - 1)
)

raw["life_bin"] = pd.cut(
    raw["relative_life"],
    bins=np.linspace(0, 1, NUMBER_OF_BINS + 1),
    labels=False,
    include_lowest=True
)


# ------------------------------------------------------------
# Calculate the starting level of each engine
#
# The starting level is the mean of its first five cycles.
# ------------------------------------------------------------

starting_levels = (
    raw.loc[
        raw["cycle"] <= STARTING_CYCLES,
        ["unit"] + SENSORS
    ]
    .groupby("unit")[SENSORS]
    .mean()
    .add_suffix("_start")
    .reset_index()
)

analysis_data = raw[
    ["unit", "cycle", "life_bin"] + SENSORS
].merge(
    starting_levels,
    on="unit",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# Standardized within-engine change
#
# Sensor value minus that engine's starting value,
# divided by the training-set standard deviation.
#
# This allows sensors with different scales to be compared.
# ------------------------------------------------------------

sensor_standard_deviation = (
    raw[SENSORS]
    .std(ddof=0)
    .replace(0, np.nan)
)

for sensor in SENSORS:

    analysis_data[sensor] = (
        analysis_data[sensor]
        - analysis_data[f"{sensor}_start"]
    ) / sensor_standard_deviation[sensor]


# ------------------------------------------------------------
# Give every engine equal weight
#
# First calculate one mean for every engine and life bin.
# Then calculate the median across the 100 engines.
# ------------------------------------------------------------

engine_bin_values = (
    analysis_data
    .groupby(
        ["unit", "life_bin"],
        observed=True
    )[SENSORS]
    .mean()
    .reset_index()
)

engines_per_bin = (
    engine_bin_values
    .groupby("life_bin")["unit"]
    .nunique()
    .reindex(range(NUMBER_OF_BINS))
)

print("Number of engines represented in each life bin:")
print(engines_per_bin.to_string())

assert engines_per_bin.eq(100).all(), (
    "Not every life bin contains all 100 engines."
)

median_trajectories = (
    engine_bin_values
    .groupby("life_bin")[SENSORS]
    .median()
    .reindex(range(NUMBER_OF_BINS))
    .T
)


# ------------------------------------------------------------
# Arrange sensors by their final change
#
# Increasing sensors appear at the top.
# Decreasing sensors appear at the bottom.
# ------------------------------------------------------------

final_bin = NUMBER_OF_BINS - 1

sensor_order = (
    median_trajectories[final_bin]
    .sort_values(ascending=False)
    .index
)

heatmap_values = median_trajectories.loc[sensor_order]


# ------------------------------------------------------------
# Use a symmetric colour scale around zero
# ------------------------------------------------------------

colour_limit = np.nanpercentile(
    np.abs(heatmap_values.to_numpy()),
    98
)

colour_limit = max(colour_limit, 0.5)

display_values = np.clip(
    heatmap_values.to_numpy(),
    -colour_limit,
    colour_limit
)

colour_norm = TwoSlopeNorm(
    vmin=-colour_limit,
    vcenter=0,
    vmax=colour_limit
)


# ------------------------------------------------------------
# Create the heatmap
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(11, 6.5)
)

image = ax.imshow(
    display_values,
    aspect="auto",
    cmap="RdBu_r",
    norm=colour_norm,
    interpolation="nearest"
)

life_labels = [
    "0–10",
    "10–20",
    "20–30",
    "30–40",
    "40–50",
    "50–60",
    "60–70",
    "70–80",
    "80–90",
    "90–100"
]

ax.set_xticks(
    np.arange(NUMBER_OF_BINS)
)

ax.set_xticklabels(
    life_labels,
    rotation=45,
    ha="right"
)

ax.set_yticks(
    np.arange(len(sensor_order))
)

ax.set_yticklabels(
    sensor_order
)

ax.set_xlabel(
    "Normalized engine life (%)"
)

ax.set_ylabel(
    "Sensor"
)


# ------------------------------------------------------------
# Add white borders between cells
# ------------------------------------------------------------

ax.set_xticks(
    np.arange(-0.5, NUMBER_OF_BINS, 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, len(sensor_order), 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linewidth=1
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)


# ------------------------------------------------------------
# Colour bar
# ------------------------------------------------------------

colour_bar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.035,
    pad=0.03
)

colour_bar.set_label(
    "Median standardized change from the engine-specific start"
)

plt.tight_layout()


# ------------------------------------------------------------
# Save the figure and numerical values
# ------------------------------------------------------------

figure_path = os.path.join(
    FIGURES_DIR,
    "fd001_sensor_trajectories_heatmap.png"
)

results_path = os.path.join(
    RESULTS_DIR,
    "fd001_sensor_trajectories_heatmap_values.csv"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

heatmap_values.columns = life_labels

heatmap_values.to_csv(
    results_path,
    index_label="sensor"
)

plt.show()
plt.close(fig)

print("\nSaved:")
print("fd001_sensor_trajectories_heatmap.png")
print("fd001_sensor_trajectories_heatmap_values.csv")